In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
customers = pd.read_csv(BASE_PATH + 'customers.csv')
invoices = pd.read_csv(BASE_PATH + 'invoices.csv')
payments = pd.read_csv(BASE_PATH + 'payments.csv')

for name, df in [('orders', so_h), ('customers', customers), ('invoices', invoices), ('payments', payments)]:
    print(name, 'rows and columns:', df.shape)
    print(list(df.columns))
    display(df.head(3))
    print()

Mounted at /content/drive
orders rows and columns: (20000, 11)
['so_id', 'customer_id', 'branch_id', 'order_date', 'delivery_date', 'order_status', 'payment_terms', 'total_order_value', 'total_gst_amount', 'grand_total', 'sales_channel']


,so_id,customer_id,branch_id,order_date,delivery_date,order_status,payment_terms,total_order_value,total_gst_amount,grand_total,sales_channel
0,SO-990591,C0070,AHM001,2022-05-24,2022-05-28,Delivered,Net 30,1345720,374439.6,1720159.6,Counter Sale
1,SO-431510,C0031,KOL001,2022-05-01,2022-05-08,Delivered,Advance,2194850,609423.0,2804273.0,Dealer Network
2,SO-495859,C0013,AHM001,2023-10-01,2023-10-02,Delivered,Advance,2620800,733824.0,3354624.0,Online



customers rows and columns: (500, 15)
['customer_id', 'customer_type', 'industry_segment', 'city', 'state', 'pincode', 'region', 'branch_id', 'credit_limit', 'current_balance', 'payment_terms', 'customer_since', 'last_purchase_date', 'total_purchase_value', 'customer_rating']


,customer_id,customer_type,industry_segment,city,state,pincode,region,branch_id,credit_limit,current_balance,payment_terms,customer_since,last_purchase_date,total_purchase_value,customer_rating
0,C0001,Retail,Construction,Kolkata,West Bengal,991035,East,KOL001,45896,43284,Advance,2024-04-10,2024-04-25,3892818,4
1,C0002,Fleet Owner,Manufacturing,Hyderabad,Telangana,464589,South,HYD001,531506,426735,Advance,2022-01-19,2024-12-04,3168730,2
2,C0003,Government,Logistics,Surat,Gujarat,202475,West,DEL001,1306933,733495,Net 30,2017-02-14,2019-06-11,3584708,2



invoices rows and columns: (18033, 10)
['invoice_id', 'so_id', 'customer_id', 'branch_id', 'invoice_date', 'due_date', 'total_order_value', 'total_gst_amount', 'grand_total', 'payment_status']


,invoice_id,so_id,customer_id,branch_id,invoice_date,due_date,total_order_value,total_gst_amount,grand_total,payment_status
0,INV-325361,SO-990591,C0070,AHM001,2022-05-28,2022-06-27,1345720,374439.6,1720159.6,Unpaid
1,INV-568476,SO-431510,C0031,KOL001,2022-05-08,2022-06-07,2194850,609423.0,2804273.0,Unpaid
2,INV-786913,SO-495859,C0013,AHM001,2023-10-02,2023-11-01,2620800,733824.0,3354624.0,Paid



payments rows and columns: (19257, 5)
['payment_id', 'invoice_id', 'payment_date', 'payment_amount', 'payment_method']


,payment_id,invoice_id,payment_date,payment_amount,payment_method
0,PAY-207775,INV-786913,2023-10-07,3354624.0,Cheque
1,PAY-942474,INV-697615,2021-11-23,1387911.0,Credit Card
2,PAY-657121,INV-747829,2024-12-26,1475.0,Bank Transfer


In [2]:
print('invoice rows:', len(invoices))
print('unique invoice IDs:', invoices['invoice_id'].nunique())
repeated_invoice_rows = invoices['invoice_id'].duplicated(keep=False)
print('invoice rows whose ID is repeated:', repeated_invoice_rows.sum())
print('invoices whose customer is not in the customers table:', (~invoices['customer_id'].isin(customers['customer_id'])).sum())
print()
print('payment rows:', len(payments))
repeated_payment_rows = payments['payment_id'].duplicated(keep=False)
print('payment rows whose payment ID is repeated:', repeated_payment_rows.sum())
print('payments whose invoice ID is not in the invoices table:', (~payments['invoice_id'].isin(invoices['invoice_id'])).sum())
print('payments that point to a repeated invoice ID:', payments['invoice_id'].isin(invoices.loc[repeated_invoice_rows, 'invoice_id']).sum())
print()
print(invoices['payment_status'].value_counts())

invoice rows: 18033
unique invoice IDs: 17836
invoice rows whose ID is repeated: 393
invoices whose customer is not in the customers table: 0

payment rows: 19257
payment rows whose payment ID is repeated: 403
payments whose invoice ID is not in the invoices table: 0
payments that point to a repeated invoice ID: 417

payment_status
Paid              12536
Partially Paid     3660
Unpaid             1837
Name: count, dtype: int64


In [3]:
clean_invoices = invoices[~repeated_invoice_rows].copy()
removed_invoice_ids = invoices.loc[repeated_invoice_rows, 'invoice_id']
points_to_repeated = payments['invoice_id'].isin(removed_invoice_ids)
clean_payments = payments[~repeated_payment_rows & ~points_to_repeated].copy()

print('clean invoices:', len(clean_invoices))
print('payments with a repeated payment ID and also pointing to a repeated invoice ID:', (repeated_payment_rows & points_to_repeated).sum())
print('payments left out for either reason:', (repeated_payment_rows | points_to_repeated).sum())
print('clean payments:', len(clean_payments))
print('clean payments that do not match a clean invoice:', (~clean_payments['invoice_id'].isin(clean_invoices['invoice_id'])).sum())
print()
print(clean_invoices['payment_status'].value_counts())

clean invoices: 17640
payments with a repeated payment ID and also pointing to a repeated invoice ID: 8
payments left out for either reason: 812
clean payments: 18445
clean payments that do not match a clean invoice: 0

payment_status
Paid              12264
Partially Paid     3578
Unpaid             1798
Name: count, dtype: int64


In [4]:
paid_per_invoice = clean_payments.groupby('invoice_id')['payment_amount'].sum()

inv_money = clean_invoices.copy()
inv_money['paid_amount'] = inv_money['invoice_id'].map(paid_per_invoice).fillna(0)
inv_money['outstanding'] = inv_money['grand_total'] - inv_money['paid_amount']

by_status = inv_money.groupby('payment_status').agg(
    invoices=('invoice_id', 'count'),
    billed=('grand_total', 'sum'),
    paid=('paid_amount', 'sum'),
    outstanding=('outstanding', 'sum')
).round(0)
print(by_status)
print()
print('all clean invoices: billed', round(inv_money['grand_total'].sum()), '| paid', round(inv_money['paid_amount'].sum()), '| outstanding', round(inv_money['outstanding'].sum()))
print()
print('Paid invoices with money still outstanding:', ((inv_money['payment_status'] == 'Paid') & (inv_money['outstanding'].abs() > 1)).sum())
print('Unpaid invoices that have a payment:', ((inv_money['payment_status'] == 'Unpaid') & (inv_money['paid_amount'] > 0)).sum())
print('Partially Paid invoices not between nothing and the full bill:', ((inv_money['payment_status'] == 'Partially Paid') & ((inv_money['paid_amount'] <= 0) | (inv_money['outstanding'] <= 1))).sum())
print('invoices paid more than billed:', (inv_money['outstanding'] < -1).sum())

                invoices        billed          paid   outstanding
payment_status                                                    
Paid               12264  1.985511e+10  1.941861e+10  4.365033e+08
Partially Paid      3578  5.826085e+09  2.871951e+09  2.954135e+09
Unpaid              1798  2.918443e+09  0.000000e+00  2.918443e+09

all clean invoices: billed 28599637861 | paid 22290557066 | outstanding 6309080795

Paid invoices with money still outstanding: 306
Unpaid invoices that have a payment: 0
Partially Paid invoices not between nothing and the full bill: 88
invoices paid more than billed: 0


In [5]:
left_out = payments[repeated_payment_rows & ~points_to_repeated]
print('payments left out only because the payment ID is repeated:', len(left_out))
print('of these, payments that point to a clean invoice:', left_out['invoice_id'].isin(clean_invoices['invoice_id']).sum())
print()

extra_paid = left_out.groupby('invoice_id')['payment_amount'].sum()
has_left_out = inv_money['invoice_id'].isin(extra_paid.index)

paid_gap = (inv_money['payment_status'] == 'Paid') & (inv_money['outstanding'].abs() > 1)
part_gap = (inv_money['payment_status'] == 'Partially Paid') & ((inv_money['paid_amount'] <= 0) | (inv_money['outstanding'] <= 1))
print('Paid invoices with money outstanding:', paid_gap.sum(), '| of these with a left-out payment:', (paid_gap & has_left_out).sum())
print('Partially Paid invoices not between nothing and the full bill:', part_gap.sum(), '| of these with a left-out payment:', (part_gap & has_left_out).sum())
print()

paid_plus = inv_money['paid_amount'] + inv_money['invoice_id'].map(extra_paid).fillna(0)
outstanding_plus = inv_money['grand_total'] - paid_plus
print('with the left-out payments added back:')
print('Paid invoices with money outstanding:', ((inv_money['payment_status'] == 'Paid') & (outstanding_plus.abs() > 1)).sum())
print('Partially Paid invoices not between nothing and the full bill:', ((inv_money['payment_status'] == 'Partially Paid') & ((paid_plus <= 0) | (outstanding_plus <= 1))).sum())
print('invoices paid more than billed:', (outstanding_plus < -1).sum())

payments left out only because the payment ID is repeated: 395
of these, payments that point to a clean invoice: 395

Paid invoices with money outstanding: 306 | of these with a left-out payment: 306
Partially Paid invoices not between nothing and the full bill: 88 | of these with a left-out payment: 88

with the left-out payments added back:
Paid invoices with money outstanding: 0
Partially Paid invoices not between nothing and the full bill: 0
invoices paid more than billed: 0


In [6]:
payments_used = payments[~points_to_repeated].copy()
print('payments used:', len(payments_used))
print('payments used that do not match a clean invoice:', (~payments_used['invoice_id'].isin(clean_invoices['invoice_id'])).sum())
print()

paid_per_invoice = payments_used.groupby('invoice_id')['payment_amount'].sum()
inv_money = clean_invoices.copy()
inv_money['paid_amount'] = inv_money['invoice_id'].map(paid_per_invoice).fillna(0)
inv_money['outstanding'] = inv_money['grand_total'] - inv_money['paid_amount']

by_status = inv_money.groupby('payment_status').agg(
    invoices=('invoice_id', 'count'),
    billed=('grand_total', 'sum'),
    paid=('paid_amount', 'sum'),
    outstanding=('outstanding', 'sum')
)
print(by_status.to_string(float_format='{:,.0f}'.format))
print()
print('all clean invoices: billed {:,.0f} | paid {:,.0f} | outstanding {:,.0f}'.format(inv_money['grand_total'].sum(), inv_money['paid_amount'].sum(), inv_money['outstanding'].sum()))
print()
print('Paid invoices with money still outstanding:', ((inv_money['payment_status'] == 'Paid') & (inv_money['outstanding'].abs() > 1)).sum())
print('Unpaid invoices that have a payment:', ((inv_money['payment_status'] == 'Unpaid') & (inv_money['paid_amount'] > 0)).sum())
print('Partially Paid invoices not between nothing and the full bill:', ((inv_money['payment_status'] == 'Partially Paid') & ((inv_money['paid_amount'] <= 0) | (inv_money['outstanding'] <= 1))).sum())
print('invoices paid more than billed:', (inv_money['outstanding'] < -1).sum())

payments used: 18840
payments used that do not match a clean invoice: 0

                invoices         billed           paid   outstanding
payment_status                                                      
Paid               12264 19,855,109,591 19,855,109,591             0
Partially Paid      3578  5,826,085,351  2,940,224,313 2,885,861,038
Unpaid              1798  2,918,442,919              0 2,918,442,919

all clean invoices: billed 28,599,637,861 | paid 22,795,333,904 | outstanding 5,804,303,957

Paid invoices with money still outstanding: 0
Unpaid invoices that have a payment: 0
Partially Paid invoices not between nothing and the full bill: 0
invoices paid more than billed: 0


In [7]:
for name, pay in [('payments used in the earlier late payment rate', clean_payments),
                  ('the 395 payments added back', left_out),
                  ('all payments now used', payments_used)]:
    merged = pay.merge(clean_invoices[['invoice_id', 'due_date']], on='invoice_id', how='inner')
    paid_on = pd.to_datetime(merged['payment_date'])
    due_on = pd.to_datetime(merged['due_date'])
    is_late = paid_on > due_on
    days_late = (paid_on - due_on).dt.days[is_late]
    print(name)
    print('  payments:', len(merged), '| late:', is_late.sum(), '| late rate: {:.2%}'.format(is_late.mean()), '| median days late:', days_late.median())

payments used in the earlier late payment rate
  payments: 18445 | late: 7655 | late rate: 41.50% | median days late: 17.0
the 395 payments added back
  payments: 395 | late: 165 | late rate: 41.77% | median days late: 16.0
all payments now used
  payments: 18840 | late: 7820 | late rate: 41.51% | median days late: 17.0


In [8]:
pay_late = payments_used.merge(
    clean_invoices[['invoice_id', 'customer_id', 'due_date']],
    on='invoice_id', how='left'
)

pay_late['payment_date'] = pd.to_datetime(pay_late['payment_date'])
pay_late['due_date'] = pd.to_datetime(pay_late['due_date'])

pay_late['days_late'] = (pay_late['payment_date'] - pay_late['due_date']).dt.days
pay_late['is_late'] = pay_late['days_late'] > 0

print("payments:", len(pay_late))
print("late payments:", int(pay_late['is_late'].sum()))
print("payments with a missing date:", int(pay_late[['payment_date', 'due_date']].isna().any(axis=1).sum()))
print("payments with no customer:", int(pay_late['customer_id'].isna().sum()))
print("average days late (late payments only):", round(pay_late.loc[pay_late['is_late'], 'days_late'].mean(), 1))

payments: 18840
late payments: 7820
payments with a missing date: 0
payments with no customer: 0
average days late (late payments only): 20.0


In [9]:
per_inv = pay_late.groupby('invoice_id').agg(
    payments=('payment_id', 'count'),
    last_payment_date=('payment_date', 'max'),
    due_date=('due_date', 'first')
).reset_index()

per_inv['last_days_late'] = (per_inv['last_payment_date'] - per_inv['due_date']).dt.days
per_inv['last_payment_late'] = per_inv['last_days_late'] > 0

inv_late = clean_invoices[['invoice_id', 'customer_id', 'payment_status']].merge(
    per_inv.drop(columns='due_date'), on='invoice_id', how='left'
)

has_payment = inv_late['payments'].notna()

print("invoices:", len(inv_late))
print("invoices with a payment:", int(has_payment.sum()))
print("invoices with no payment:", int((~has_payment).sum()))
print("status of invoices with no payment:")
print(inv_late.loc[~has_payment, 'payment_status'].value_counts())
print("invoices with 2 or more payments:", int((inv_late['payments'] >= 2).sum()))
print("invoices whose last payment was late:", int(inv_late['last_payment_late'].sum()))

invoices: 17640
invoices with a payment: 15842
invoices with no payment: 1798
status of invoices with no payment:
payment_status
Unpaid    1798
Name: count, dtype: int64
invoices with 2 or more payments: 2368
invoices whose last payment was late: 7180


In [10]:
inv_late['last_payment_late'] = inv_late['last_payment_late'].fillna(False).astype(bool)

by_billed = clean_invoices.groupby('customer_id').agg(
    invoices=('invoice_id', 'count'),
    billed=('grand_total', 'sum')
)

by_paid = pay_late.groupby('customer_id').agg(
    paid=('payment_amount', 'sum'),
    late_payments=('is_late', 'sum')
)

by_status = pd.crosstab(clean_invoices['customer_id'], clean_invoices['payment_status'])
by_status.columns = [c.lower().replace(' ', '_') + '_invoices' for c in by_status.columns]

by_late = inv_late.groupby('customer_id').agg(
    invoices_with_payment=('payments', lambda s: int(s.notna().sum())),
    late_invoices=('last_payment_late', 'sum')
)

customer_risk = by_billed.join(by_paid).join(by_status).join(by_late).fillna(0)
customer_risk['outstanding'] = customer_risk['billed'] - customer_risk['paid']
customer_risk['late_invoice_rate'] = (
    customer_risk['late_invoices'] / customer_risk['invoices_with_payment'].replace(0, float('nan'))
)

print("customers in the table:", len(customer_risk))
print("billed total:", int(customer_risk['billed'].sum()), "| difference to 28599637861:", int(customer_risk['billed'].sum()) - 28599637861)
print("paid total:", int(customer_risk['paid'].sum()), "| difference to 22795333904:", int(customer_risk['paid'].sum()) - 22795333904)
print("outstanding total:", int(customer_risk['outstanding'].sum()), "| difference to 5804303957:", int(customer_risk['outstanding'].sum()) - 5804303957)
print("late invoice rate (all customers together):", round(customer_risk['late_invoices'].sum() / customer_risk['invoices_with_payment'].sum() * 100, 2), "%")
print("customers with no payment at all:", int((customer_risk['invoices_with_payment'] == 0).sum()))
print()
print(customer_risk.sort_values('outstanding', ascending=False).head(5)[['invoices', 'billed', 'paid', 'outstanding', 'late_invoice_rate']])

customers in the table: 500
billed total: 28599637860 | difference to 28599637861: -1
paid total: 22795333903 | difference to 22795333904: -1
outstanding total: 5804303956 | difference to 5804303957: -1
late invoice rate (all customers together): 45.32 %
customers with no payment at all: 0

             invoices      billed         paid  outstanding  late_invoice_rate
customer_id                                                                   
C0045              42  75275350.6  44826515.59  30448835.01           0.166667
C0297              32  68772003.6  40130815.38  28641188.22           0.960000
C0319              33  60426895.0  33637532.52  26789362.48           0.541667
C0421              38  69466242.8  43728858.21  25737384.59           0.533333
C0135              35  62791824.6  37263038.81  25528785.79           0.071429


/tmp/ipykernel_3494/4088970873.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  inv_late['last_payment_late'] = inv_late['last_payment_late'].fillna(False).astype(bool)


In [11]:
print("billed: customer table", round(customer_risk['billed'].sum(), 2), "| invoices table", round(clean_invoices['grand_total'].sum(), 2), "| difference", round(customer_risk['billed'].sum() - clean_invoices['grand_total'].sum(), 2))
print("paid: customer table", round(customer_risk['paid'].sum(), 2), "| payments table", round(pay_late['payment_amount'].sum(), 2), "| difference", round(customer_risk['paid'].sum() - pay_late['payment_amount'].sum(), 2))
print("outstanding: customer table", round(customer_risk['outstanding'].sum(), 2), "| billed minus paid", round(clean_invoices['grand_total'].sum() - pay_late['payment_amount'].sum(), 2), "| difference", round(customer_risk['outstanding'].sum() - (clean_invoices['grand_total'].sum() - pay_late['payment_amount'].sum()), 2))

billed: customer table 28599637860.6 | invoices table 28599637860.6 | difference -0.0
paid: customer table 22795333903.87 | payments table 22795333903.87 | difference 0.0
outstanding: customer table 5804303956.73 | billed minus paid 5804303956.73 | difference -0.0


In [12]:
delivered = so_h[so_h['order_status'] == 'Delivered']
value_kpi15 = delivered.groupby('customer_id')['total_order_value'].sum().rename('value_before_gst')

customer_risk = customer_risk.drop(columns=['value_before_gst', 'value_rank'], errors='ignore').join(value_kpi15)
customer_risk['value_rank'] = customer_risk['value_before_gst'].rank(ascending=False, method='first').astype(int)

print("customers with a money value:", int(customer_risk['value_before_gst'].notna().sum()))
print("money added over customers:", int(customer_risk['value_before_gst'].sum()), "| difference to 22958987430:", int(customer_risk['value_before_gst'].sum()) - 22958987430)

top = customer_risk.sort_values('value_rank').iloc[0]
bottom = customer_risk.sort_values('value_rank').iloc[-1]
print("rank 1:", customer_risk['value_rank'].idxmin(), int(top['value_before_gst']))
print("rank 500:", customer_risk['value_rank'].idxmax(), int(bottom['value_before_gst']))

clean_before_gst = clean_invoices['total_order_value'].sum()
print("clean invoices before GST:", int(clean_before_gst), "| gap to 22958987430:", int(22958987430 - clean_before_gst))
print("billed incl. GST divided by billed before GST:", round(clean_invoices['grand_total'].sum() / clean_before_gst, 4))

customers with a money value: 500
money added over customers: 22958987430 | difference to 22958987430: 0
rank 1: C0153 84966970
rank 500: C0347 17721530
clean invoices before GST: 22445461170 | gap to 22958987430: 513526260
billed incl. GST divided by billed before GST: 1.2742


In [13]:
invoiced_orders = set(clean_invoices['so_id'])
no_clean_invoice = delivered[~delivered['so_id'].isin(invoiced_orders)]

print("delivered orders:", len(delivered))
print("delivered orders with a clean invoice:", int(delivered['so_id'].isin(invoiced_orders).sum()))
print("delivered orders with no clean invoice:", len(no_clean_invoice))
print("their money before GST:", int(no_clean_invoice['total_order_value'].sum()), "| gap seen before: 513526260")
print("customers touched:", no_clean_invoice['customer_id'].nunique())
print("most such orders for one customer:", int(no_clean_invoice.groupby('customer_id').size().max()))
print("clean invoices whose order is not delivered:", int((~clean_invoices['so_id'].isin(set(delivered['so_id']))).sum()))

delivered orders: 18033
delivered orders with a clean invoice: 17640
delivered orders with no clean invoice: 393
their money before GST: 513526260 | gap seen before: 513526260
customers touched: 284
most such orders for one customer: 3
clean invoices whose order is not delivered: 0


In [14]:
from scipy.stats import binomtest

late_trials = customer_risk['invoices_with_payment'].astype(int)
late_hits = customer_risk['late_invoices'].astype(int)
p_late = late_hits.sum() / late_trials.sum()

unpaid_trials = customer_risk['invoices'].astype(int)
unpaid_hits = customer_risk['unpaid_invoices'].astype(int)
p_unpaid = unpaid_hits.sum() / unpaid_trials.sum()

def spread_report(name, hits, trials, p):
    rate = hits / trials
    real_spread = rate.std()
    luck_spread = ((p * (1 - p)) / trials).mean() ** 0.5
    pvalues = [binomtest(int(h), int(n), p).pvalue for h, n in zip(hits, trials)]
    pvalues = pd.Series(pvalues, index=hits.index)
    print(name)
    print("  overall rate:", round(p * 100, 2), "%")
    print("  real spread between customers:", round(real_spread, 4))
    print("  spread expected from luck alone:", round(luck_spread, 4))
    print("  customers with p below 0.05:", int((pvalues < 0.05).sum()), "(about 25 expected by luck)")
    print("  customers with p below 0.0001:", int((pvalues < 0.0001).sum()))
    return pvalues

late_p = spread_report("late invoice rate", late_hits, late_trials, p_late)
unpaid_p = spread_report("unpaid invoice share", unpaid_hits, unpaid_trials, p_unpaid)

late invoice rate
  overall rate: 45.32 %
  real spread between customers: 0.2568
  spread expected from luck alone: 0.0899
  customers with p below 0.05: 265 (about 25 expected by luck)
  customers with p below 0.0001: 130
unpaid invoice share
  overall rate: 10.19 %
  real spread between customers: 0.0528
  spread expected from luck alone: 0.0517
  customers with p below 0.05: 20 (about 25 expected by luck)
  customers with p below 0.0001: 0


In [15]:
band_edges = [-0.001, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
band_names = ['0-10%', '10-20%', '20-30%', '30-40%', '40-50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']

late_band = pd.cut(customer_risk['late_invoice_rate'], bins=band_edges, labels=band_names)

band_table = customer_risk.groupby(late_band, observed=False).agg(
    customers=('invoices', 'size'),
    invoices=('invoices', 'sum'),
    billed=('billed', 'sum'),
    outstanding=('outstanding', 'sum')
)
band_table['billed'] = band_table['billed'].round(0).astype('int64')
band_table['outstanding'] = band_table['outstanding'].round(0).astype('int64')

print(band_table.to_string())
print()
print("customers added:", int(band_table['customers'].sum()), "| invoices added:", int(band_table['invoices'].sum()))
print("customers with a late rate missing:", int(customer_risk['late_invoice_rate'].isna().sum()))
print("lowest late rate:", round(customer_risk['late_invoice_rate'].min(), 3), "| middle:", round(customer_risk['late_invoice_rate'].median(), 3), "| highest:", round(customer_risk['late_invoice_rate'].max(), 3))

                   customers  invoices      billed  outstanding
late_invoice_rate                                              
0-10%                     83      2868  4773268078    941034025
10-20%                    30      1054  1666290392    373772806
20-30%                    25       872  1389157345    251655876
30-40%                    60      2110  3440116289    709816665
40-50%                    60      2104  3412036871    712661474
50-60%                    81      2882  4621875337    919703324
60-70%                    69      2498  4023378174    735504616
70-80%                    58      2049  3372049510    701154842
80-90%                    28      1027  1586894333    390300529
90-100%                    6       176   314571532     68699799

customers added: 500 | invoices added: 17640
customers with a late rate missing: 0
lowest late rate: 0.0 | middle: 0.493 | highest: 1.0


In [16]:
terms_source = clean_invoices[['invoice_id', 'invoice_date', 'due_date']].copy()
terms_source['terms_days'] = (pd.to_datetime(terms_source['due_date']) - pd.to_datetime(terms_source['invoice_date'])).dt.days

inv_terms = inv_late.merge(terms_source[['invoice_id', 'terms_days']], on='invoice_id', how='left')
with_payment = inv_terms[inv_terms['payments'].notna()]

terms_table = with_payment.groupby('terms_days').agg(
    invoices=('invoice_id', 'count'),
    late_invoices=('last_payment_late', 'sum')
)
terms_table['late_rate_percent'] = (terms_table['late_invoices'] / terms_table['invoices'] * 100).round(2)

print(terms_table.to_string())
print()
print("invoices with a payment:", len(with_payment), "(expected 15842)")
print("invoices with a missing due period:", int(inv_terms['terms_days'].isna().sum()))
print("customers with more than one due period length:", int((inv_terms.groupby('customer_id')['terms_days'].nunique() > 1).sum()))

            invoices  late_invoices  late_rate_percent
terms_days                                            
15              2772           2178              78.57
30              6195           3556              57.40
45              3549           1208              34.04
60              3326            238               7.16

invoices with a payment: 15842 (expected 15842)
invoices with a missing due period: 0
customers with more than one due period length: 0


In [17]:
customer_terms = inv_terms.groupby('customer_id')['terms_days'].first()
customer_risk['terms_days'] = customer_risk.index.map(customer_terms)

print("customers with a due period:", int(customer_risk['terms_days'].notna().sum()))
print()

for days, group in customer_risk.groupby('terms_days'):
    hits = group['late_invoices'].astype(int)
    trials = group['invoices_with_payment'].astype(int)
    p_group = hits.sum() / trials.sum()
    real_spread = (hits / trials).std()
    luck_spread = ((p_group * (1 - p_group)) / trials).mean() ** 0.5
    pvalues = pd.Series([binomtest(int(h), int(n), p_group).pvalue for h, n in zip(hits, trials)])
    print(int(days), "day due period")
    print("  customers:", len(group), "| late rate:", round(p_group * 100, 2), "%")
    print("  real spread between customers:", round(real_spread, 4), "| spread from luck alone:", round(luck_spread, 4))
    print("  customers with p below 0.05:", int((pvalues < 0.05).sum()), "(about", round(len(group) * 0.05, 1), "expected by luck)")
    print("  customers with p below 0.0001:", int((pvalues < 0.0001).sum()))

customers with a due period: 500

15 day due period
  customers: 87 | late rate: 78.57 %
  real spread between customers: 0.0767 | spread from luck alone: 0.074
  customers with p below 0.05: 4 (about 4.4 expected by luck)
  customers with p below 0.0001: 0
30 day due period
  customers: 193 | late rate: 57.4 %
  real spread between customers: 0.09 | spread from luck alone: 0.0887
  customers with p below 0.05: 9 (about 9.7 expected by luck)
  customers with p below 0.0001: 0
45 day due period
  customers: 113 | late rate: 34.04 %
  real spread between customers: 0.09 | spread from luck alone: 0.086
  customers with p below 0.05: 7 (about 5.7 expected by luck)
  customers with p below 0.0001: 0
60 day due period
  customers: 107 | late rate: 7.16 %
  real spread between customers: 0.0496 | spread from luck alone: 0.047
  customers with p below 0.05: 5 (about 5.4 expected by luck)
  customers with p below 0.0001: 0


In [18]:
invoice_dates = clean_invoices[['invoice_id', 'invoice_date']]
pace = with_payment.merge(invoice_dates, on='invoice_id', how='left')
pace['days_to_pay'] = (pd.to_datetime(pace['last_payment_date']) - pd.to_datetime(pace['invoice_date'])).dt.days

pace_table = pace.groupby('terms_days').agg(
    invoices=('invoice_id', 'count'),
    median_days_to_pay=('days_to_pay', 'median'),
    average_days_to_pay=('days_to_pay', 'mean'),
    fewest_days=('days_to_pay', 'min'),
    most_days=('days_to_pay', 'max')
)
pace_table['average_days_to_pay'] = pace_table['average_days_to_pay'].round(1)

print(pace_table.to_string())
print()
print("invoices in the table:", int(pace_table['invoices'].sum()), "(expected 15842)")
print("payments dated before the invoice date:", int((pace['days_to_pay'] < 0).sum()))

            invoices  median_days_to_pay  average_days_to_pay  fewest_days  most_days
terms_days                                                                           
15              2772                34.0                 34.5            0         90
30              6195                35.0                 35.2            0         90
45              3549                35.0                 35.3            0         90
60              3326                36.0                 35.4            0         90

invoices in the table: 15842 (expected 15842)
payments dated before the invoice date: 0


In [19]:
from scipy.stats import spearmanr

total_outstanding = customer_risk['outstanding'].sum()
total_billed = customer_risk['billed'].sum()
sorted_outstanding = customer_risk['outstanding'].sort_values(ascending=False)

print("customers:", len(customer_risk), "| outstanding total:", round(total_outstanding, 2))
for n in (10, 50, 100):
    share = sorted_outstanding.head(n).sum() / total_outstanding * 100
    print("top", n, "customers by outstanding hold", round(share, 2), "% of it (an even spread would be", n / 5, "%)")
print()

is_valuable = customer_risk['value_rank'] <= 100
for label, mask in [("100 most valuable customers", is_valuable), ("other 400 customers", ~is_valuable)]:
    part = customer_risk[mask]
    print(label)
    print("  customers:", len(part), "| billed:", round(part['billed'].sum(), 2), "| outstanding:", round(part['outstanding'].sum(), 2))
    print("  share of all billing:", round(part['billed'].sum() / total_billed * 100, 2), "% | share of all outstanding:", round(part['outstanding'].sum() / total_outstanding * 100, 2), "%")
    print("  outstanding as share of their own billing:", round(part['outstanding'].sum() / part['billed'].sum() * 100, 2), "%")

outstanding_share = customer_risk['outstanding'] / customer_risk['billed']
corr = spearmanr(customer_risk['value_rank'], outstanding_share)
print()
print("rank correlation of value rank and outstanding share of billing:", round(corr.correlation, 3), "| p-value:", round(corr.pvalue, 4))

customers: 500 | outstanding total: 5804303956.73
top 10 customers by outstanding hold 4.48 % of it (an even spread would be 2.0 %)
top 50 customers by outstanding hold 18.38 % of it (an even spread would be 10.0 %)
top 100 customers by outstanding hold 32.83 % of it (an even spread would be 20.0 %)

100 most valuable customers
  customers: 100 | billed: 7648156846.4 | outstanding: 1510885496.85
  share of all billing: 26.74 % | share of all outstanding: 26.03 %
  outstanding as share of their own billing: 19.75 %
other 400 customers
  customers: 400 | billed: 20951481014.2 | outstanding: 4293418459.88
  share of all billing: 73.26 % | share of all outstanding: 73.97 %
  outstanding as share of their own billing: 20.49 %

rank correlation of value rank and outstanding share of billing: 0.042 | p-value: 0.3444


In [20]:
valuable_one = customer_risk[customer_risk['value_rank'] <= 100]
priority_list = valuable_one.sort_values('outstanding', ascending=False).head(20)

show = priority_list[['value_rank', 'billed', 'outstanding', 'unpaid_invoices', 'partially_paid_invoices', 'terms_days']].copy()
show['unpaid_share_of_billing_percent'] = (priority_list['outstanding'] / priority_list['billed'] * 100).round(1)
show['billed'] = show['billed'].round(0).astype('int64')
show['outstanding'] = show['outstanding'].round(0).astype('int64')
print(show.to_string())

print()
print("these 20 customers hold:", round(priority_list['outstanding'].sum(), 2), "which is", round(priority_list['outstanding'].sum() / customer_risk['outstanding'].sum() * 100, 2), "% of all outstanding money")

valuable_two_ids = set(customer_risk['billed'].nlargest(100).index)
valuable_one_ids = set(valuable_one.index)
print("customers valuable under rule one:", len(valuable_one_ids), "| under rule two:", len(valuable_two_ids))
print("customers valuable under both rules:", len(valuable_one_ids & valuable_two_ids))
print("outstanding held by the rule two group:", round(customer_risk.loc[list(valuable_two_ids), 'outstanding'].sum(), 2))

             value_rank     billed  outstanding  unpaid_invoices  partially_paid_invoices  terms_days  unpaid_share_of_billing_percent
customer_id                                                                                                                           
C0045                52   75275351     30448835                6                       14          60                             40.4
C0297                68   68772004     28641188                7                       11          15                             41.6
C0421                96   69466243     25737385                8                        7          30                             37.1
C0207                14   83475657     25369012                7                       12          30                             30.4
C0288                 2   97218404     24774937                6                       11          30                             25.5
C0475                 4   94883375     24558266        

In [21]:
details = customers.set_index('customer_id')[['customer_type', 'branch_id', 'region', 'industry_segment']]
cut = customer_risk.join(details)
cut['open_invoices'] = cut['unpaid_invoices'] + cut['partially_paid_invoices']

overall_open_rate = cut['open_invoices'].sum() / cut['invoices'].sum()
all_outstanding = cut['outstanding'].sum()

print("customers without details:", int(cut['customer_type'].isna().sum()))
print("overall share of invoices not fully paid:", round(overall_open_rate * 100, 2), "% | strict line for the p-value:", round(0.05 / 27, 4))

for column in ['terms_days', 'customer_type', 'branch_id', 'region', 'industry_segment']:
    group = cut.groupby(column).agg(
        customers=('invoices', 'size'),
        invoices=('invoices', 'sum'),
        open_invoices=('open_invoices', 'sum'),
        billed=('billed', 'sum'),
        outstanding=('outstanding', 'sum')
    )
    group['outstanding_percent_of_billing'] = (group['outstanding'] / group['billed'] * 100).round(2)
    group['share_of_all_outstanding_percent'] = (group['outstanding'] / all_outstanding * 100).round(2)
    group['not_fully_paid_percent'] = (group['open_invoices'] / group['invoices'] * 100).round(2)
    group['p_value'] = [round(binomtest(int(o), int(n), overall_open_rate).pvalue, 4) for o, n in zip(group['open_invoices'], group['invoices'])]
    group['billed'] = group['billed'].round(0).astype('int64')
    group['outstanding'] = group['outstanding'].round(0).astype('int64')
    print()
    print(column)
    print(group.to_string())
    print("  customers added:", int(group['customers'].sum()), "| outstanding added:", int(group['outstanding'].sum()))

customers without details: 0
overall share of invoices not fully paid: 30.48 % | strict line for the p-value: 0.0019

terms_days
            customers  invoices  open_invoices       billed  outstanding  outstanding_percent_of_billing  share_of_all_outstanding_percent  not_fully_paid_percent  p_value
terms_days                                                                                                                                                                 
15                 87      3094            948   4982798598   1054341481                           21.16                             18.16                   30.64   0.8452
30                193      6858           2031  11058093373   2181376525                           19.73                             37.58                   29.62   0.1217
45                113      3967           1221   6422965018   1320936435                           20.57                             22.76                   30.78   0.6790
60         

In [22]:
paid_by_invoice = payments_used.groupby('invoice_id')['payment_amount'].sum()

ageing = clean_invoices[['invoice_id', 'customer_id', 'invoice_date', 'grand_total', 'payment_status']].copy()
ageing['invoice_date'] = pd.to_datetime(ageing['invoice_date'])
ageing['paid'] = ageing['invoice_id'].map(paid_by_invoice).fillna(0)
ageing['outstanding'] = ageing['grand_total'] - ageing['paid']

open_invoices = ageing[ageing['outstanding'] > 1].copy()

latest_date = max(ageing['invoice_date'].max(), pd.to_datetime(payments_used['payment_date']).max())
snapshot = latest_date + pd.Timedelta(days=1)
open_invoices['age_days'] = (snapshot - open_invoices['invoice_date']).dt.days

age_edges = [-1, 30, 90, 180, 365, 100000]
age_names = ['0-30 days', '31-90 days', '91-180 days', '181-365 days', 'over 365 days']
open_invoices['age_band'] = pd.cut(open_invoices['age_days'], bins=age_edges, labels=age_names)

age_table = open_invoices.groupby('age_band', observed=False).agg(
    invoices=('invoice_id', 'count'),
    outstanding=('outstanding', 'sum')
)
age_table['share_of_outstanding_percent'] = (age_table['outstanding'] / open_invoices['outstanding'].sum() * 100).round(2)
age_table['outstanding'] = age_table['outstanding'].round(0).astype('int64')

print("latest date in invoices and payments:", latest_date.date(), "| today used:", snapshot.date())
print("open invoices:", len(open_invoices), "(expected 5376)")
print("open invoices by status:", open_invoices['payment_status'].value_counts().to_dict())
print("outstanding of open invoices:", round(open_invoices['outstanding'].sum(), 2), "(expected 5804303956.73)")
print("oldest open invoice (days):", int(open_invoices['age_days'].max()), "| middle:", int(open_invoices['age_days'].median()))
print()
print(age_table.to_string())

latest date in invoices and payments: 2025-03-20 | today used: 2025-03-21
open invoices: 5376 (expected 5376)
open invoices by status: {'Partially Paid': 3578, 'Unpaid': 1798}
outstanding of open invoices: 5804303956.56 (expected 5804303956.73)
oldest open invoice (days): 2268 | middle: 1182

               invoices  outstanding  share_of_outstanding_percent
age_band                                                          
0-30 days             0            0                          0.00
31-90 days           41     38263531                          0.66
91-180 days         209    216057037                          3.72
181-365 days        416    441440520                          7.61
over 365 days      4710   5108542868                         88.01


In [23]:
ageing['invoice_year'] = ageing['invoice_date'].dt.year

year_table = ageing.groupby('invoice_year').agg(
    invoices=('invoice_id', 'count'),
    billed=('grand_total', 'sum'),
    outstanding=('outstanding', 'sum'),
    open_invoices=('outstanding', lambda s: int((s > 1).sum()))
)
year_table['outstanding_percent_of_billing'] = (year_table['outstanding'] / year_table['billed'] * 100).round(2)
year_table['open_invoice_percent'] = (year_table['open_invoices'] / year_table['invoices'] * 100).round(2)
year_table['billed'] = year_table['billed'].round(0).astype('int64')
year_table['outstanding'] = year_table['outstanding'].round(0).astype('int64')
print(year_table.to_string())

print()
print("invoices added:", int(year_table['invoices'].sum()), "| first invoice date:", ageing['invoice_date'].min().date(), "| last invoice date:", ageing['invoice_date'].max().date())
print("billed added:", round(ageing['grand_total'].sum(), 2), "| outstanding added:", round(ageing['outstanding'].sum(), 2))

small = ageing[(ageing['outstanding'] > -1) & (ageing['outstanding'] <= 1)]
print("invoices with 1 or less outstanding:", len(small), "| their outstanding added:", round(small['outstanding'].sum(), 2))
print("invoices paid more than 1 over the bill:", int((ageing['outstanding'] < -1).sum()))

              invoices      billed  outstanding  open_invoices  outstanding_percent_of_billing  open_invoice_percent
invoice_year                                                                                                        
2019              2911  4771199102    914664887            865                           19.17                 29.71
2020              2970  4863343984   1046512696            943                           21.52                 31.75
2021              2952  4757345306    949467312            903                           19.96                 30.59
2022              2885  4780701603   1008071697            884                           21.09                 30.64
2023              2975  4789278680    956718676            901                           19.98                 30.29
2024              2900  4553561874    915144399            868                           20.10                 29.93
2025                47    84207312     13724290             12  

In [24]:
tables_to_check = {
    "clean_invoices": clean_invoices,
    "payments_used": payments_used,
    "customers": customers,
    "customer_risk": customer_risk,
}

for name, table in tables_to_check.items():
    text_columns = table.select_dtypes(include="object")
    empty_text = text_columns.apply(lambda c: c.astype(str).str.strip().eq("")).sum()
    blank = table.isna().sum().add(empty_text, fill_value=0).astype(int)

    numbers = table.select_dtypes(include="number")
    zero = (numbers == 0).sum()
    negative = (numbers < 0).sum()

    print("Table:", name, "| rows:", len(table))
    print("Total blank cells:", int(blank.sum()))
    print("Total negative numbers:", int(negative.sum()))
    print("Columns with zeros:")
    print(zero[zero > 0].to_string() if (zero > 0).any() else "none")
    print("Columns with blanks:")
    print(blank[blank > 0].to_string() if (blank > 0).any() else "none")
    print("Columns with negatives:")
    print(negative[negative > 0].to_string() if (negative > 0).any() else "none")
    print("-" * 40)

Table: clean_invoices | rows: 17640
Total blank cells: 0
Total negative numbers: 0
Columns with zeros:
none
Columns with blanks:
none
Columns with negatives:
none
----------------------------------------
Table: payments_used | rows: 18840
Total blank cells: 0
Total negative numbers: 0
Columns with zeros:
none
Columns with blanks:
none
Columns with negatives:
none
----------------------------------------
Table: customers | rows: 500
Total blank cells: 0
Total negative numbers: 0
Columns with zeros:
none
Columns with blanks:
none
Columns with negatives:
none
----------------------------------------
Table: customer_risk | rows: 500
Total blank cells: 0
Total negative numbers: 0
Columns with zeros:
late_payments        16
unpaid_invoices      13
late_invoices        16
late_invoice_rate    16
Columns with blanks:
none
Columns with negatives:
none
----------------------------------------


In [25]:
invoice_dates = pd.to_datetime(clean_invoices["invoice_date"])
due_dates = pd.to_datetime(clean_invoices["due_date"])

print("Customers in the customers table:", customers["customer_id"].nunique())
print("Customers with at least one clean invoice:", clean_invoices["customer_id"].nunique())
print("Customers with no clean invoice:", len(set(customers["customer_id"]) - set(clean_invoices["customer_id"])))

payments_per_invoice = payments_used.groupby("invoice_id").size()
print("Invoices with 2 or more payments:", int((payments_per_invoice >= 2).sum()))
print("Most payments on one invoice:", int(payments_per_invoice.max()))

print("Invoices with a due date before the invoice date:", int((due_dates < invoice_dates).sum()))
print("Invoices with a due date on the invoice day:", int((due_dates == invoice_dates).sum()))

payment_check = payments_used.merge(
    clean_invoices[["invoice_id"]].assign(invoice_date=invoice_dates.values),
    on="invoice_id",
)
payment_check["payment_date"] = pd.to_datetime(payment_check["payment_date"])
print("Payments made on the invoice day:", int((payment_check["payment_date"] == payment_check["invoice_date"]).sum()))
print("Payments made before the invoice date:", int((payment_check["payment_date"] < payment_check["invoice_date"]).sum()))

due_period = (due_dates - invoice_dates).dt.days
per_customer = clean_invoices.assign(due_period=due_period.values).groupby("customer_id")["due_period"]
print("Most due period lengths for one customer:", int(per_customer.nunique().max()))
print("Customers per due period:")
print(per_customer.first().value_counts().sort_index().to_string())

no_late_payments = customer_risk["late_payments"] == 0
no_late_invoices = customer_risk["late_invoices"] == 0
print("Customers with no late payments:", int(no_late_payments.sum()))
print("Customers with no late invoices:", int(no_late_invoices.sum()))
print("Customers with both:", int((no_late_payments & no_late_invoices).sum()))

Customers in the customers table: 500
Customers with at least one clean invoice: 500
Customers with no clean invoice: 0
Invoices with 2 or more payments: 2368
Most payments on one invoice: 3
Invoices with a due date before the invoice date: 0
Invoices with a due date on the invoice day: 0
Payments made on the invoice day: 292
Payments made before the invoice date: 0
Most due period lengths for one customer: 1
Customers per due period:
due_period
15     87
30    193
45    113
60    107
Customers with no late payments: 16
Customers with no late invoices: 16
Customers with both: 16


In [26]:
print("Invoices with a repeated invoice ID:", int(clean_invoices["invoice_id"].duplicated(keep=False).sum()))
print("Invoices with a repeated order ID:", int(clean_invoices["so_id"].duplicated(keep=False).sum()))

payment_columns = list(payments.columns)
payments_now = payments_used[payment_columns]
repeat_id_rows = payments_now[payments_now["payment_id"].duplicated(keep=False)]

print("Payment rows used:", len(payments_now))
print("Payment rows that share a payment ID with another used row:", len(repeat_id_rows))
print("Payment IDs involved:", repeat_id_rows["payment_id"].nunique())

print("Payment rows that are exact copies of another row (every column the same):", int(payments_now.duplicated(keep=False).sum()))

other_columns = [c for c in payment_columns if c != "payment_id"]
print("Payment rows with the same invoice, date, amount and method under a different payment ID:", int(payments_now.duplicated(subset=other_columns, keep=False).sum()))

invoices_per_id = repeat_id_rows.groupby("payment_id")["invoice_id"].nunique()
print("Payment IDs whose rows point to different invoices:", int((invoices_per_id > 1).sum()))
print("Payment IDs whose rows all point to the same invoice:", int((invoices_per_id == 1).sum()))

Invoices with a repeated invoice ID: 0
Invoices with a repeated order ID: 0
Payment rows used: 18840
Payment rows that share a payment ID with another used row: 387
Payment IDs involved: 193
Payment rows that are exact copies of another row (every column the same): 0
Payment rows with the same invoice, date, amount and method under a different payment ID: 0
Payment IDs whose rows point to different invoices: 193
Payment IDs whose rows all point to the same invoice: 0


In [27]:
billed_invoices = clean_invoices["grand_total"].sum()
paid_payments = payments_used["payment_amount"].sum()
print("Billed in the invoice table:", f"{billed_invoices:,.2f}")
print("Billed in the customer table:", f"{customer_risk['billed'].sum():,.2f}")
print("Billed difference:", round(billed_invoices - customer_risk["billed"].sum(), 2))
print("Paid in the payment table:", f"{paid_payments:,.2f}")
print("Paid in the customer table:", f"{customer_risk['paid'].sum():,.2f}")
print("Paid difference:", round(paid_payments - customer_risk["paid"].sum(), 2))
print("Outstanding in the customer table:", f"{customer_risk['outstanding'].sum():,.2f}")
print("Outstanding difference (billed minus paid minus outstanding):", round(billed_invoices - paid_payments - customer_risk["outstanding"].sum(), 2))

paid_by_invoice = payments_used.groupby("invoice_id")["payment_amount"].sum()
by_status = clean_invoices[["invoice_id", "payment_status", "grand_total"]].copy()
by_status["paid"] = by_status["invoice_id"].map(paid_by_invoice).fillna(0)
by_status["outstanding"] = by_status["grand_total"] - by_status["paid"]
status_table = by_status.groupby("payment_status").agg(
    invoices=("invoice_id", "count"),
    billed=("grand_total", "sum"),
    paid=("paid", "sum"),
    outstanding=("outstanding", "sum"),
).round(2)
print("Money by payment status:")
print(status_table.to_string())

late_count = int(pay_late["is_late"].sum())
on_time_count = int((~pay_late["is_late"].astype(bool)).sum())
print("Late payments:", late_count, "| on time payments:", on_time_count, "| together:", late_count + on_time_count)

risk = customer_risk.set_index("customer_id") if "customer_id" in customer_risk.columns else customer_risk
before_gst_clean = clean_invoices.groupby("customer_id")["total_order_value"].sum().reindex(risk.index).fillna(0)
gap_per_customer = risk["value_before_gst"] - before_gst_clean
print("Clean invoices before GST:", f"{before_gst_clean.sum():,.0f}")
print("Customer value before GST (KPI 15 money):", f"{risk['value_before_gst'].sum():,.0f}")
print("Gap added over customers:", f"{gap_per_customer.sum():,.0f}")
print("Customers with a gap:", int((gap_per_customer > 0.5).sum()))
print("Customers with no gap:", int((gap_per_customer.abs() <= 0.5).sum()))
print("Customers with a negative gap:", int((gap_per_customer < -0.5).sum()))

Billed in the invoice table: 28,599,637,860.60
Billed in the customer table: 28,599,637,860.60
Billed difference: 0.0
Paid in the payment table: 22,795,333,903.87
Paid in the customer table: 22,795,333,903.87
Paid difference: 0.0
Outstanding in the customer table: 5,804,303,956.73
Outstanding difference (billed minus paid minus outstanding): 0.0
Money by payment status:
                invoices        billed          paid   outstanding
payment_status                                                    
Paid               12264  1.985511e+10  1.985511e+10  1.700000e-01
Partially Paid      3578  5.826085e+09  2.940224e+09  2.885861e+09
Unpaid              1798  2.918443e+09  0.000000e+00  2.918443e+09
Late payments: 7820 | on time payments: 11020 | together: 18840
Clean invoices before GST: 22,445,461,170
Customer value before GST (KPI 15 money): 22,958,987,430
Gap added over customers: 513,526,260
Customers with a gap: 284
Customers with no gap: 216
Customers with a negative gap: 0


In [28]:
import csv

with open(BASE_PATH + "invoices.csv", newline="") as f:
    raw_invoices = list(csv.DictReader(f))
with open(BASE_PATH + "payments.csv", newline="") as f:
    raw_payments = list(csv.DictReader(f))

id_counts = {}
for row in raw_invoices:
    id_counts[row["invoice_id"]] = id_counts.get(row["invoice_id"], 0) + 1
repeated_ids = {k for k, v in id_counts.items() if v > 1}

loop_invoices = {}
for row in raw_invoices:
    if row["invoice_id"] not in repeated_ids:
        loop_invoices[row["invoice_id"]] = {
            "customer": row["customer_id"],
            "invoice_date": pd.Timestamp(row["invoice_date"]),
            "due": pd.Timestamp(row["due_date"]),
            "total": float(row["grand_total"]),
        }

loop_customers = {}
for inv in loop_invoices.values():
    c = loop_customers.setdefault(inv["customer"], {"invoices": 0, "billed": 0.0, "paid": 0.0, "late_payments": 0, "late_invoices": 0})
    c["invoices"] += 1
    c["billed"] += inv["total"]

paid_by_invoice_loop = {}
last_payment_loop = {}
latest_payment_date = None
for row in raw_payments:
    invoice_id = row["invoice_id"]
    if invoice_id in repeated_ids or invoice_id not in loop_invoices:
        continue
    pay_date = pd.Timestamp(row["payment_date"])
    amount = float(row["payment_amount"])
    inv = loop_invoices[invoice_id]
    c = loop_customers[inv["customer"]]
    c["paid"] += amount
    if pay_date > inv["due"]:
        c["late_payments"] += 1
    paid_by_invoice_loop[invoice_id] = paid_by_invoice_loop.get(invoice_id, 0.0) + amount
    if invoice_id not in last_payment_loop or pay_date > last_payment_loop[invoice_id]:
        last_payment_loop[invoice_id] = pay_date
    if latest_payment_date is None or pay_date > latest_payment_date:
        latest_payment_date = pay_date

for invoice_id, last_date in last_payment_loop.items():
    inv = loop_invoices[invoice_id]
    if last_date > inv["due"]:
        loop_customers[inv["customer"]]["late_invoices"] += 1

loop_table = pd.DataFrame.from_dict(loop_customers, orient="index")
loop_table["outstanding"] = loop_table["billed"] - loop_table["paid"]

risk = customer_risk.set_index("customer_id") if "customer_id" in customer_risk.columns else customer_risk
print("Customers in the loop table:", len(loop_table), "| in the customer table:", len(risk))
print("Customers in only one of the two:", len(set(loop_table.index) ^ set(risk.index)))

same_order = loop_table.reindex(risk.index)
for name in ["invoices", "late_payments", "late_invoices"]:
    print("Customers where", name, "differ:", int((same_order[name] != risk[name]).sum()))
for name in ["billed", "paid", "outstanding"]:
    print("Customers where", name, "differ by more than one cent:", int(((same_order[name] - risk[name]).abs() > 0.01).sum()))

with_payment = [(i, d) for i, d in last_payment_loop.items()]
terms_rows = []
for invoice_id, last_date in with_payment:
    inv = loop_invoices[invoice_id]
    terms_rows.append(((inv["due"] - inv["invoice_date"]).days, last_date > inv["due"]))
terms_check = pd.DataFrame(terms_rows, columns=["due_period", "late"]).groupby("due_period")["late"].agg(["count", "sum"])
terms_check.columns = ["invoices_with_payment", "late_invoices"]
print("Late invoices by due period (second method):")
print(terms_check.to_string())

latest_invoice_date = max(inv["invoice_date"] for inv in loop_invoices.values())
today = max(latest_payment_date, latest_invoice_date) + pd.Timedelta(days=1)
print("Today used:", today.date())
band_edges = [(0, 30), (31, 90), (91, 180), (181, 365), (100000, 100000)]
band_names = ["0 to 30 days", "31 to 90 days", "91 to 180 days", "181 to 365 days", "over 365 days"]
band_counts = [0] * 5
band_money = [0.0] * 5
open_total = 0
for invoice_id, inv in loop_invoices.items():
    left = inv["total"] - paid_by_invoice_loop.get(invoice_id, 0.0)
    if left <= 1:
        continue
    open_total += 1
    age = (today - inv["invoice_date"]).days
    if age <= 30:
        slot = 0
    elif age <= 90:
        slot = 1
    elif age <= 180:
        slot = 2
    elif age <= 365:
        slot = 3
    else:
        slot = 4
    band_counts[slot] += 1
    band_money[slot] += left
print("Open invoices (second method):", open_total)
for name, count, money in zip(band_names, band_counts, band_money):
    print(name, "|", count, "invoices |", f"{money:,.0f}")

Customers in the loop table: 500 | in the customer table: 500
Customers in only one of the two: 0
Customers where invoices differ: 0
Customers where late_payments differ: 0
Customers where late_invoices differ: 0
Customers where billed differ by more than one cent: 0
Customers where paid differ by more than one cent: 0
Customers where outstanding differ by more than one cent: 0
Late invoices by due period (second method):
            invoices_with_payment  late_invoices
due_period                                      
15                           2772           2178
30                           6195           3556
45                           3549           1208
60                           3326            238
Today used: 2025-03-21
Open invoices (second method): 5376
0 to 30 days | 0 invoices | 0
31 to 90 days | 41 invoices | 38,263,531
91 to 180 days | 209 invoices | 216,057,037
181 to 365 days | 416 invoices | 441,440,520
over 365 days | 4710 invoices | 5,108,542,868


In [29]:
invoice_table = clean_invoices[["invoice_id", "customer_id", "grand_total"]].copy()
invoice_table["paid"] = invoice_table["invoice_id"].map(payments_used.groupby("invoice_id")["payment_amount"].sum()).fillna(0)
invoice_table["open_amount"] = (invoice_table["grand_total"] - invoice_table["paid"]).clip(lower=0)
invoice_table.loc[invoice_table["open_amount"] <= 1, "open_amount"] = 0

print("Invoice size: average", f"{clean_invoices['grand_total'].mean():,.0f}", "| middle", f"{clean_invoices['grand_total'].median():,.0f}", "| largest", f"{clean_invoices['grand_total'].max():,.0f}")
print("Largest invoice is", round(clean_invoices["grand_total"].max() / clean_invoices["grand_total"].mean(), 1), "times the average")
print("Payment size: average", f"{payments_used['payment_amount'].mean():,.0f}", "| middle", f"{payments_used['payment_amount'].median():,.0f}", "| largest", f"{payments_used['payment_amount'].max():,.0f}")
print("Largest payment is", round(payments_used["payment_amount"].max() / payments_used["payment_amount"].mean(), 1), "times the average")

payment_vs_invoice = payments_used.merge(clean_invoices[["invoice_id", "grand_total"]], on="invoice_id")
print("Payments bigger than their own invoice:", int((payment_vs_invoice["payment_amount"] > payment_vs_invoice["grand_total"] + 0.01).sum()))

risk = customer_risk.set_index("customer_id") if "customer_id" in customer_risk.columns else customer_risk
print("Outstanding as a share of billed, all customers:", round(risk["outstanding"].sum() / risk["billed"].sum() * 100, 2), "%")
for n in [10, 50, 100]:
    rest = risk.drop(risk["billed"].nlargest(n).index)
    print("Without the", n, "customers with the most billed:", round(rest["outstanding"].sum() / rest["billed"].sum() * 100, 2), "%")
rest = risk.drop(risk["outstanding"].nlargest(10).index)
print("Without the 10 customers with the most outstanding:", round(rest["outstanding"].sum() / rest["billed"].sum() * 100, 2), "%")

largest_open = invoice_table.groupby("customer_id")["open_amount"].max().reindex(risk.index)
without_largest = risk["outstanding"] - largest_open
valuable = risk[risk["value_rank"] <= 100]
original_top = valuable["outstanding"].nlargest(20).index
new_top = without_largest.loc[valuable.index].nlargest(20).index
print("Priority customers still in the top 20 after removing each customer's largest open invoice:", len(set(original_top) & set(new_top)), "of 20")
share_in_largest = largest_open.loc[original_top] / valuable.loc[original_top, "outstanding"]
print("Share of a priority customer's outstanding sitting in one invoice: average", round(share_in_largest.mean() * 100, 1), "% | highest", round(share_in_largest.max() * 100, 1), "%")
print("Outstanding of the 20 priority customers:", f"{valuable.loc[original_top, 'outstanding'].sum():,.0f}", "| after removing their largest open invoices:", f"{without_largest.loc[original_top].sum():,.0f}")

Invoice size: average 1,621,295 | middle 1,285,161 | largest 10,451,982
Largest invoice is 6.4 times the average
Payment size: average 1,209,943 | middle 820,354 | largest 9,970,301
Largest payment is 8.2 times the average
Payments bigger than their own invoice: 0
Outstanding as a share of billed, all customers: 20.3 %
Without the 10 customers with the most billed: 20.36 %
Without the 50 customers with the most billed: 20.43 %
Without the 100 customers with the most billed: 20.53 %
Without the 10 customers with the most outstanding: 19.91 %
Priority customers still in the top 20 after removing each customer's largest open invoice: 16 of 20
Share of a priority customer's outstanding sitting in one invoice: average 20.5 % | highest 32.8 %
Outstanding of the 20 priority customers: 461,987,109 | after removing their largest open invoices: 368,081,554


In [30]:
invoice_dates = pd.to_datetime(clean_invoices["invoice_date"])
months = invoice_dates.dt.to_period("M").value_counts().sort_index()
print("First invoice:", invoice_dates.min().date(), "| last invoice:", invoice_dates.max().date())
print("Months with invoices:", len(months))
print("Empty months between the first and last month:", len(pd.period_range(months.index.min(), months.index.max(), freq="M")) - len(months))
full_months = months.iloc[:-1]
print("Invoices per month (without the last, partial month): fewest", int(full_months.min()), "| middle", int(full_months.median()), "| most", int(full_months.max()))
print("Invoices in the last month", str(months.index[-1]), ":", int(months.iloc[-1]))

payment_dates = pd.to_datetime(pay_late["payment_date"])
print("First payment:", payment_dates.min().date(), "| last payment:", payment_dates.max().date())
late_by_year = pay_late.assign(year=payment_dates.dt.year).groupby("year")["is_late"].agg(["count", "mean"])
late_by_year["late_percent"] = (late_by_year["mean"] * 100).round(2)
print("Payments and late rate by payment year:")
print(late_by_year[["count", "late_percent"]].to_string())

invoice_history = invoice_table.merge(clean_invoices[["invoice_id", "invoice_date"]], on="invoice_id")
invoice_history["invoice_date"] = pd.to_datetime(invoice_history["invoice_date"])
cutoff = payment_dates.max() - pd.Timedelta(days=90)
print("Cutoff for invoices with a full 90 day payment window:", cutoff.date())
for name, part in [("up to the cutoff", invoice_history[invoice_history["invoice_date"] <= cutoff]),
                   ("after the cutoff", invoice_history[invoice_history["invoice_date"] > cutoff])]:
    share = part["open_amount"].sum() / part["grand_total"].sum() * 100
    print("Invoices", name, ":", len(part), "| billed", f"{part['grand_total'].sum():,.0f}", "| outstanding share", round(share, 2), "%")

open_invoices_history = invoice_history[invoice_history["open_amount"] > 0]
for extra_days in [0, 30, 365]:
    today_moved = payment_dates.max() + pd.Timedelta(days=1 + extra_days)
    age = (today_moved - open_invoices_history["invoice_date"]).dt.days
    total_open = open_invoices_history["open_amount"].sum()
    older_90 = open_invoices_history.loc[age > 90, "open_amount"].sum() / total_open * 100
    older_365 = open_invoices_history.loc[age > 365, "open_amount"].sum() / total_open * 100
    print("Today moved by", extra_days, "days:", round(older_90, 2), "% older than 90 days |", round(older_365, 2), "% older than a year")

First invoice: 2019-01-03 | last invoice: 2025-01-13
Months with invoices: 73
Empty months between the first and last month: 0
Invoices per month (without the last, partial month): fewest 170 | middle 244 | most 285
Invoices in the last month 2025-01 : 47
First payment: 2019-01-09 | last payment: 2025-03-20
Payments and late rate by payment year:
      count  late_percent
year                     
2019   2808         39.78
2020   3122         41.42
2021   3155         40.03
2022   3171         43.20
2023   3172         41.20
2024   3070         41.40
2025    342         58.19
Cutoff for invoices with a full 90 day payment window: 2024-12-20
Invoices up to the cutoff : 17504 | billed 28,370,323,313 | outstanding share 20.32 %
Invoices after the cutoff : 136 | billed 229,314,547 | outstanding share 16.69 %
Today moved by 0 days: 99.34 % older than 90 days | 88.01 % older than a year
Today moved by 30 days: 100.0 % older than 90 days | 89.34 % older than a year
Today moved by 365 days: 10

In [31]:
print("Five months with the fewest invoices (without the last, partial month):")
print(months.iloc[:-1].nsmallest(5).to_string())

invoice_date_lookup = clean_invoices.set_index("invoice_id")["invoice_date"]
payment_months = pay_late.copy()
payment_months["payment_date_value"] = pd.to_datetime(payment_months["payment_date"])
payment_months["invoice_date_value"] = pd.to_datetime(payment_months["invoice_id"].map(invoice_date_lookup))
payment_months["days_to_pay"] = (payment_months["payment_date_value"] - payment_months["invoice_date_value"]).dt.days
payment_months["month"] = payment_months["payment_date_value"].dt.to_period("M")

recent = payment_months[payment_months["month"] >= pd.Period("2024-07", freq="M")]
recent_table = recent.groupby("month").agg(
    payments=("is_late", "count"),
    late_percent=("is_late", lambda s: round(s.mean() * 100, 2)),
    median_days_to_pay=("days_to_pay", "median"),
)
print("Payments by month from July 2024:")
print(recent_table.to_string())

Five months with the fewest invoices (without the last, partial month):
invoice_date
2019-01    170
2023-02    212
2020-04    213
2019-02    214
2020-02    217
Freq: M
Payments by month from July 2024:
         payments  late_percent  median_days_to_pay
month                                              
2024-07       240         35.42                29.0
2024-08       244         39.75                32.0
2024-09       240         47.50                34.5
2024-10       310         40.65                32.0
2024-11       237         45.57                37.0
2024-12       286         37.06                32.0
2025-01       234         50.85                39.5
2025-02        91         71.43                52.0
2025-03        17         88.24                60.0


In [32]:
def show_customer(customer_id):
    risk_table = customer_risk.set_index("customer_id") if "customer_id" in customer_risk.columns else customer_risk
    customer_rows = [r for r in raw_invoices if r["customer_id"] == customer_id]
    left_out = [r for r in customer_rows if r["invoice_id"] in repeated_ids]
    used = [r for r in customer_rows if r["invoice_id"] not in repeated_ids]
    print("Customer", customer_id, "| invoice rows in the raw file:", len(customer_rows), "| left out for a repeated invoice ID:", len(left_out), "| used:", len(used))
    print("invoice_id | invoice date | due date | total | payments | paid | last payment | outcome | raw status")

    billed = paid_total = 0.0
    late_payments = late_invoices = 0
    for r in sorted(used, key=lambda row: row["invoice_date"]):
        payments_here = [p for p in raw_payments if p["invoice_id"] == r["invoice_id"]]
        total = float(r["grand_total"])
        paid_here = sum(float(p["payment_amount"]) for p in payments_here)
        due = pd.Timestamp(r["due_date"])
        late_payments += sum(1 for p in payments_here if pd.Timestamp(p["payment_date"]) > due)
        last_payment = max((p["payment_date"] for p in payments_here), default="none")
        if payments_here:
            outcome = "late" if pd.Timestamp(last_payment) > due else "on time"
            late_invoices += 1 if outcome == "late" else 0
        else:
            outcome = "no payment"
        billed += total
        paid_total += paid_here
        print(r["invoice_id"], "|", r["invoice_date"], "|", r["due_date"], "|", f"{total:,.2f}", "|", len(payments_here), "|", f"{paid_here:,.2f}", "|", last_payment, "|", outcome, "|", r["payment_status"])

    print("From the raw rows: invoices", len(used), "| billed", f"{billed:,.2f}", "| paid", f"{paid_total:,.2f}", "| outstanding", f"{billed - paid_total:,.2f}", "| late payments", late_payments, "| late invoices", late_invoices)
    row = risk_table.loc[customer_id]
    print("From the customer table: invoices", int(row["invoices"]), "| billed", f"{row['billed']:,.2f}", "| paid", f"{row['paid']:,.2f}", "| outstanding", f"{row['outstanding']:,.2f}", "| late payments", int(row["late_payments"]), "| late invoices", int(row["late_invoices"]))

show_customer("C0045")

Customer C0045 | invoice rows in the raw file: 42 | left out for a repeated invoice ID: 0 | used: 42
invoice_id | invoice date | due date | total | payments | paid | last payment | outcome | raw status
INV-353919 | 2019-02-08 | 2019-04-09 | 2,951,711.20 | 1 | 1,308,183.74 | 2019-04-14 | late | Partially Paid
INV-445564 | 2019-02-20 | 2019-04-21 | 2,642,069.60 | 1 | 2,642,069.60 | 2019-04-06 | on time | Paid
INV-628881 | 2019-03-25 | 2019-05-24 | 3,602,978.20 | 1 | 1,275,096.94 | 2019-04-10 | on time | Partially Paid
INV-897641 | 2019-06-26 | 2019-08-25 | 1,200,987.00 | 1 | 1,200,987.00 | 2019-07-08 | on time | Paid
INV-612746 | 2019-07-24 | 2019-09-22 | 403,112.00 | 0 | 0.00 | none | no payment | Unpaid
INV-106295 | 2019-08-15 | 2019-10-14 | 2,755,494.60 | 1 | 2,755,494.60 | 2019-09-22 | on time | Paid
INV-970301 | 2019-10-02 | 2019-12-01 | 1,395,968.00 | 1 | 1,395,968.00 | 2019-10-17 | on time | Paid
INV-216934 | 2019-10-21 | 2019-12-20 | 718,733.00 | 1 | 718,733.00 | 2019-11-16 | on 

In [33]:
from collections import Counter

def show_customer_short(customer_id):
    risk_table = customer_risk.set_index("customer_id") if "customer_id" in customer_risk.columns else customer_risk
    customer_rows = [r for r in raw_invoices if r["customer_id"] == customer_id]
    left_out = [r for r in customer_rows if r["invoice_id"] in repeated_ids]
    used = [r for r in customer_rows if r["invoice_id"] not in repeated_ids]
    print("Customer", customer_id, "| invoice rows in the raw file:", len(customer_rows), "| left out for a repeated invoice ID:", len(left_out), "| used:", len(used))
    for r in left_out:
        print("Left out:", r["invoice_id"], r["invoice_date"], f"{float(r['grand_total']):,.2f}", r["payment_status"])

    billed = paid_total = 0.0
    late_payments = late_invoices = status_mismatches = 0
    raw_status_counts = Counter(r["payment_status"] for r in used)
    print("Invoices not fully paid: invoice_id | due date | total | paid | remaining | outcome | raw status")
    for r in sorted(used, key=lambda row: row["invoice_date"]):
        payments_here = [p for p in raw_payments if p["invoice_id"] == r["invoice_id"]]
        total = float(r["grand_total"])
        paid_here = sum(float(p["payment_amount"]) for p in payments_here)
        remaining = total - paid_here
        due = pd.Timestamp(r["due_date"])
        late_payments += sum(1 for p in payments_here if pd.Timestamp(p["payment_date"]) > due)
        if payments_here:
            last_payment = max(pd.Timestamp(p["payment_date"]) for p in payments_here)
            outcome = "late" if last_payment > due else "on time"
            late_invoices += 1 if outcome == "late" else 0
        else:
            outcome = "no payment"
        worked_out_status = "Unpaid" if paid_here == 0 else ("Paid" if remaining <= 1 else "Partially Paid")
        status_mismatches += 1 if worked_out_status != r["payment_status"] else 0
        billed += total
        paid_total += paid_here
        if remaining > 1:
            print(r["invoice_id"], "|", r["due_date"], "|", f"{total:,.2f}", "|", f"{paid_here:,.2f}", "|", f"{remaining:,.2f}", "|", outcome, "|", r["payment_status"])

    print("Raw status counts:", dict(raw_status_counts), "| invoices whose status does not match their payments:", status_mismatches)
    print("From the raw rows: invoices", len(used), "| billed", f"{billed:,.2f}", "| paid", f"{paid_total:,.2f}", "| outstanding", f"{billed - paid_total:,.2f}", "| late payments", late_payments, "| late invoices", late_invoices)
    row = risk_table.loc[customer_id]
    print("From the customer table: invoices", int(row["invoices"]), "| billed", f"{row['billed']:,.2f}", "| paid", f"{row['paid']:,.2f}", "| outstanding", f"{row['outstanding']:,.2f}", "| late payments", int(row["late_payments"]), "| late invoices", int(row["late_invoices"]))
    print("-" * 40)

left_out_counts = Counter(r["customer_id"] for r in raw_invoices if r["invoice_id"] in repeated_ids)
chosen_customer = sorted(left_out_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
print("Customer chosen because it has the most left-out invoices:", chosen_customer)
print("-" * 40)

show_customer_short("C0297")
show_customer_short(chosen_customer)

Customer chosen because it has the most left-out invoices: C0006
----------------------------------------
Customer C0297 | invoice rows in the raw file: 34 | left out for a repeated invoice ID: 2 | used: 32
Left out: INV-509998 2024-04-03 642,464.00 Paid
Left out: INV-628478 2021-05-31 3,650,548.00 Partially Paid
Invoices not fully paid: invoice_id | due date | total | paid | remaining | outcome | raw status
INV-401604 | 2019-03-15 | 660,101.00 | 261,906.35 | 398,194.65 | late | Partially Paid
INV-437984 | 2019-11-27 | 3,347,074.20 | 945,729.26 | 2,401,344.94 | late | Partially Paid
INV-975026 | 2019-12-05 | 881,087.20 | 424,390.17 | 456,697.03 | on time | Partially Paid
INV-631512 | 2020-11-27 | 3,564,612.20 | 2,126,978.33 | 1,437,633.87 | late | Partially Paid
INV-775627 | 2021-03-22 | 4,354,565.60 | 0.00 | 4,354,565.60 | no payment | Unpaid
INV-427643 | 2021-04-12 | 1,384,083.20 | 0.00 | 1,384,083.20 | no payment | Unpaid
INV-502484 | 2021-07-18 | 4,798,399.00 | 0.00 | 4,798,399.00 

In [34]:
band_edges = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
band_names = ["0 to 10%", "10 to 20%", "20 to 30%", "30 to 40%", "40 to 50%", "50 to 60%", "60 to 70%", "70 to 80%", "80 to 90%", "90 to 100%"]
late_band = pd.cut(customer_risk["late_invoice_rate"], bins=band_edges, labels=band_names, include_lowest=True)

table_by_period = pd.crosstab(late_band, customer_risk["terms_days"], margins=True, margins_name="All customers")
print("Customers by late rate band and due period (days):")
print(table_by_period.to_string())

Customers by late rate band and due period (days):
terms_days         15   30   45   60  All customers
late_invoice_rate                                  
0 to 10%            0    0    0   83             83
10 to 20%           0    0    8   22             30
20 to 30%           0    0   23    2             25
30 to 40%           0    6   54    0             60
40 to 50%           0   36   24    0             60
50 to 60%           1   76    4    0             81
60 to 70%           8   61    0    0             69
70 to 80%          45   13    0    0             58
80 to 90%          27    1    0    0             28
90 to 100%          6    0    0    0              6
All customers      87  193  113  107            500


In [35]:
original_rank = valuable["outstanding"].rank(ascending=False, method="first").astype(int)
new_rank = without_largest.loc[valuable.index].rank(ascending=False, method="first").astype(int)

dropped_out = [c for c in original_top if c not in set(new_top)]
came_in = [c for c in new_top if c not in set(original_top)]

def show_changes(title, customers):
    print(title)
    for c in customers:
        print(c, "| original rank", int(original_rank[c]), "| rank without largest open invoice", int(new_rank[c]),
              "| outstanding", f"{valuable.loc[c, 'outstanding']:,.0f}",
              "| largest open invoice", f"{largest_open[c]:,.0f}",
              "| outstanding without it", f"{without_largest[c]:,.0f}")

show_changes("Left the top 20:", dropped_out)
show_changes("Came into the top 20:", came_in)

original_sorted = valuable["outstanding"].sort_values(ascending=False)
new_sorted = without_largest.loc[valuable.index].sort_values(ascending=False)
print("Original list: 20th customer", f"{original_sorted.iloc[19]:,.0f}", "| 21st customer", f"{original_sorted.iloc[20]:,.0f}")
print("List without largest open invoices: 20th customer", f"{new_sorted.iloc[19]:,.0f}", "| 21st customer", f"{new_sorted.iloc[20]:,.0f}")

Left the top 20:
C0084 | original rank 15 | rank without largest open invoice 29 | outstanding 20,945,200 | largest open invoice 6,879,455 | outstanding without it 14,065,745
C0060 | original rank 17 | rank without largest open invoice 22 | outstanding 20,019,107 | largest open invoice 4,795,829 | outstanding without it 15,223,277
C0482 | original rank 18 | rank without largest open invoice 25 | outstanding 20,014,759 | largest open invoice 5,128,822 | outstanding without it 14,885,937
C0272 | original rank 19 | rank without largest open invoice 26 | outstanding 19,883,826 | largest open invoice 5,120,711 | outstanding without it 14,763,115
Came into the top 20:
C0201 | original rank 22 | rank without largest open invoice 15 | outstanding 19,510,311 | largest open invoice 2,722,408 | outstanding without it 16,787,903
C0300 | original rank 26 | rank without largest open invoice 18 | outstanding 18,383,128 | largest open invoice 2,667,406 | outstanding without it 15,715,722
C0384 | origi

In [36]:
from scipy.stats import spearmanr

risk_table = customer_risk.set_index("customer_id") if "customer_id" in customer_risk.columns else customer_risk
customer_columns = customers.set_index("customer_id")[["credit_limit", "current_balance"]].copy()
for name in ["credit_limit", "current_balance"]:
    converted = pd.to_numeric(customer_columns[name], errors="coerce")
    print(name, "| values that could not be read as numbers:", int(converted.isna().sum()))
    customer_columns[name] = converted

compare = customer_columns.join(risk_table[["billed", "outstanding"]])
print("Customers compared:", len(compare))
print(compare[["credit_limit", "current_balance", "billed", "outstanding"]].describe().round(0).to_string())

print("Total current_balance:", f"{compare['current_balance'].sum():,.0f}", "| total outstanding we calculated:", f"{compare['outstanding'].sum():,.0f}")
difference = (compare["current_balance"] - compare["outstanding"]).abs()
print("Customers where current_balance is within 1 of outstanding:", int((difference <= 1).sum()))
print("Customers where current_balance is within 10% of outstanding:", int((difference <= 0.1 * compare["outstanding"].clip(lower=1)).sum()))
print("Rank correlation of current_balance and outstanding:", round(spearmanr(compare["current_balance"], compare["outstanding"])[0], 3))

print("Customers whose outstanding is above their credit_limit:", int((compare["outstanding"] > compare["credit_limit"]).sum()))
print("Customers whose billed money is above their credit_limit:", int((compare["billed"] > compare["credit_limit"]).sum()))
print("Rank correlation of credit_limit and billed:", round(spearmanr(compare["credit_limit"], compare["billed"])[0], 3))

credit_limit | values that could not be read as numbers: 0
current_balance | values that could not be read as numbers: 0
Customers compared: 500
       credit_limit  current_balance       billed  outstanding
count         500.0            500.0        500.0        500.0
mean       491525.0         246681.0   57199276.0   11608608.0
std        372084.0         258848.0   13153390.0    4860350.0
min         10552.0             63.0   22536660.0    1205247.0
25%        210644.0          37390.0   47571805.0    7922430.0
50%        417145.0         178570.0   56651611.0   10955701.0
75%        713694.0         348900.0   65273826.0   14604553.0
max       1493816.0        1435105.0  103578606.0   30448835.0
Total current_balance: 123,340,274 | total outstanding we calculated: 5,804,303,957
Customers where current_balance is within 1 of outstanding: 0
Customers where current_balance is within 10% of outstanding: 0
Rank correlation of current_balance and outstanding: -0.006
Customers whose ou

In [37]:
risk_table = customer_risk.set_index("customer_id") if "customer_id" in customer_risk.columns else customer_risk
customer_terms = customers.set_index("customer_id")["payment_terms"]

print("Customers: payment terms in the customers table against the due period on their invoices (days)")
print(pd.crosstab(customer_terms.reindex(risk_table.index), risk_table["terms_days"], margins=True, margins_name="All customers").to_string())

orders_terms = so_h[["so_id", "customer_id", "payment_terms"]].copy()
orders_terms["customer_terms"] = orders_terms["customer_id"].map(customer_terms)
print("Orders whose payment terms differ from the customer's payment terms:", int((orders_terms["payment_terms"] != orders_terms["customer_terms"]).sum()), "of", len(orders_terms))

invoice_days = clean_invoices[["so_id", "invoice_date", "due_date"]].copy()
invoice_days["due_period_days"] = (pd.to_datetime(invoice_days["due_date"]) - pd.to_datetime(invoice_days["invoice_date"])).dt.days
invoice_terms = invoice_days.merge(orders_terms[["so_id", "payment_terms"]], on="so_id", how="left")
print("Invoices whose order has no payment terms:", int(invoice_terms["payment_terms"].isna().sum()))
print("Invoices: payment terms on the order against the due period on the invoice (days)")
print(pd.crosstab(invoice_terms["payment_terms"], invoice_terms["due_period_days"], margins=True, margins_name="All invoices").to_string())

Customers: payment terms in the customers table against the due period on their invoices (days)
terms_days     15   30   45   60  All customers
payment_terms                                  
Advance         0  103    0    0            103
Net 15         87    0    0    0             87
Net 30          0   90    0    0             90
Net 45          0    0  113    0            113
Net 60          0    0    0  107            107
All customers  87  193  113  107            500
Orders whose payment terms differ from the customer's payment terms: 0 of 20000
Invoices whose order has no payment terms: 0
Invoices: payment terms on the order against the due period on the invoice (days)
due_period_days    15    30    45    60  All invoices
payment_terms                                        
Advance             0  3661     0     0          3661
Net 15           3094     0     0     0          3094
Net 30              0  3197     0     0          3197
Net 45              0     0  3967     0    

In [38]:
total_out = customer_risk['outstanding'].sum()
total_billed = customer_risk['billed'].sum()

def gini(values):
    x = values.sort_values().reset_index(drop=True)
    i = x.index + 1
    return (2 * (i * x).sum()) / (len(x) * x.sum()) - (len(x) + 1) / len(x)

by_out = customer_risk.sort_values('outstanding', ascending=False)
by_billed = customer_risk.sort_values('billed', ascending=False)

print("Customers:", len(customer_risk))
print("Total outstanding:", round(total_out, 2))
print("Total billed:", round(total_billed, 2))
print()

for n in [10, 50, 100]:
    even = n / len(customer_risk) * 100
    a = by_out.head(n)
    b = by_billed.head(n)
    print(f"Biggest {n} customers (an even spread would be {even:.1f}%)")
    print(f"  Most outstanding: hold {a['outstanding'].sum() / total_out * 100:.2f}% of outstanding and {a['billed'].sum() / total_billed * 100:.2f}% of billing")
    print(f"  Most billed: hold {b['billed'].sum() / total_billed * 100:.2f}% of billing and {b['outstanding'].sum() / total_out * 100:.2f}% of outstanding")
    print()

print("Gini of outstanding money:", round(gini(customer_risk['outstanding']), 4))
print("Gini of billed money:", round(gini(customer_risk['billed']), 4))

Customers: 500
Total outstanding: 5804303956.73
Total billed: 28599637860.6

Biggest 10 customers (an even spread would be 2.0%)
  Most outstanding: hold 4.48% of outstanding and 2.62% of billing
  Most billed: hold 3.27% of billing and 2.97% of outstanding

Biggest 50 customers (an even spread would be 10.0%)
  Most outstanding: hold 18.38% of outstanding and 12.42% of billing
  Most billed: hold 14.49% of billing and 13.94% of outstanding

Biggest 100 customers (an even spread would be 20.0%)
  Most outstanding: hold 32.83% of outstanding and 23.41% of billing
  Most billed: hold 26.80% of billing and 25.97% of outstanding

Gini of outstanding money: 0.2328
Gini of billed money: 0.1285


In [39]:
inv = inv_money[['customer_id', 'grand_total', 'outstanding']].copy()
inv['share_unpaid'] = inv['outstanding'] / inv['grand_total']

def concentration(out_by_customer):
    s = out_by_customer.sort_values(ascending=False)
    tot = s.sum()
    return [s.head(10).sum() / tot * 100,
            s.head(50).sum() / tot * 100,
            s.head(100).sum() / tot * 100,
            gini(out_by_customer)]

real = concentration(inv.groupby('customer_id')['outstanding'].sum())

runs = []
for seed in range(500):
    shuffled = inv['share_unpaid'].sample(frac=1, random_state=seed).values
    luck_out = pd.Series(shuffled * inv['grand_total'].values,
                         index=inv['customer_id'].values).groupby(level=0).sum()
    runs.append(concentration(luck_out))

luck = pd.DataFrame(runs, columns=['top 10 share', 'top 50 share', 'top 100 share', 'gini'])

print("Invoices:", len(inv), " Customers:", inv['customer_id'].nunique())
print("Shuffles run:", len(luck))
print()
for i, name in enumerate(luck.columns):
    col = luck[name]
    print(name)
    print(f"  real: {real[i]:.4f}")
    print(f"  luck: average {col.mean():.4f}, lowest {col.min():.4f}, highest {col.max():.4f}")
    print(f"  shuffles as high as the real number: {(col >= real[i]).sum()} of {len(luck)}")
    print()

Invoices: 17640  Customers: 500
Shuffles run: 500

top 10 share
  real: 4.4831
  luck: average 4.5656, lowest 3.9272, highest 5.2647
  shuffles as high as the real number: 325 of 500

top 50 share
  real: 18.3816
  luck: average 18.6553, lowest 17.2054, highest 20.2833
  shuffles as high as the real number: 379 of 500

top 100 share
  real: 32.8251
  luck: average 33.2102, lowest 31.5319, highest 34.9850
  shuffles as high as the real number: 393 of 500

gini
  real: 0.2328
  luck: average 0.2431, lowest 0.2177, highest 0.2652
  shuffles as high as the real number: 462 of 500



In [40]:
from scipy.stats import fisher_exact, mannwhitneyu

terms = so_h[['so_id', 'payment_terms']].rename(columns={'payment_terms': 'order_terms'})
d = clean_invoices[['invoice_id', 'so_id', 'grand_total', 'invoice_date', 'due_date']].merge(terms, on='so_id', how='left')
d = d[d['order_terms'].isin(['Advance', 'Net 30'])].copy()
d['invoice_date'] = pd.to_datetime(d['invoice_date'])
d['due_date'] = pd.to_datetime(d['due_date'])

pay = payments_used.copy()
pay['payment_date'] = pd.to_datetime(pay['payment_date'])
g = pay.groupby('invoice_id').agg(first_pay=('payment_date', 'min'),
                                  last_pay=('payment_date', 'max'),
                                  paid=('payment_amount', 'sum'))
d = d.merge(g, on='invoice_id', how='left')
d['paid'] = d['paid'].fillna(0)
d['outstanding'] = d['grand_total'] - d['paid']
d['has_payment'] = d['last_pay'].notna()
d['late'] = d['last_pay'] > d['due_date']
d['days_to_last'] = (d['last_pay'] - d['invoice_date']).dt.days
d['first_gap'] = (d['first_pay'] - d['invoice_date']).dt.days
d['not_fully_paid'] = d['outstanding'] > 1

for name in ['Advance', 'Net 30']:
    x = d[d['order_terms'] == name]
    p = x[x['has_payment']]
    print(name)
    print("  invoices:", len(x), " customers:", so_h[so_h['payment_terms'] == name]['customer_id'].nunique())
    print("  invoices with a payment:", len(p))
    print(f"  late invoice rate (of invoices with a payment): {p['late'].mean() * 100:.2f}%")
    print(f"  days from invoice to last payment: median {p['days_to_last'].median()}, average {p['days_to_last'].mean():.1f}")
    print("  first payment before the invoice date:", (p['first_gap'] < 0).sum(), " on the invoice day:", (p['first_gap'] == 0).sum())
    print(f"  invoices not fully paid: {x['not_fully_paid'].mean() * 100:.2f}%")
    print(f"  outstanding as % of billed: {x['outstanding'].sum() / x['grand_total'].sum() * 100:.2f}%")
    print()

adv = d[d['order_terms'] == 'Advance']
net = d[d['order_terms'] == 'Net 30']
adv_p = adv[adv['has_payment']]
net_p = net[net['has_payment']]

table_late = [[int(adv_p['late'].sum()), int((~adv_p['late']).sum())],
              [int(net_p['late'].sum()), int((~net_p['late']).sum())]]
table_open = [[int(adv['not_fully_paid'].sum()), int((~adv['not_fully_paid']).sum())],
              [int(net['not_fully_paid'].sum()), int((~net['not_fully_paid']).sum())]]

print("p-value, late rate Advance vs Net 30:", round(fisher_exact(table_late)[1], 4))
print("p-value, not fully paid Advance vs Net 30:", round(fisher_exact(table_open)[1], 4))
print("p-value, days to last payment Advance vs Net 30:", round(mannwhitneyu(adv_p['days_to_last'], net_p['days_to_last']).pvalue, 4))

Advance
  invoices: 3661  customers: 103
  invoices with a payment: 3334
  late invoice rate (of invoices with a payment): 57.35%
  days from invoice to last payment: median 35.0, average 35.3
  first payment before the invoice date: 0  on the invoice day: 70
  invoices not fully paid: 29.25%
  outstanding as % of billed: 19.32%

Net 30
  invoices: 3197  customers: 90
  invoices with a payment: 2861
  late invoice rate (of invoices with a payment): 57.46%
  days from invoice to last payment: median 36.0, average 35.1
  first payment before the invoice date: 0  on the invoice day: 65
  invoices not fully paid: 30.03%
  outstanding as % of billed: 20.19%

p-value, late rate Advance vs Net 30: 0.9384
p-value, not fully paid Advance vs Net 30: 0.4907
p-value, days to last payment Advance vs Net 30: 0.681


In [41]:
from scipy.stats import spearmanr

cr = customer_risk.copy()
cr['outstanding_pct'] = cr['outstanding'] / cr['billed'] * 100

r, p = spearmanr(cr['late_invoice_rate'], cr['outstanding_pct'])
print("All customers:", len(cr), " rank correlation:", round(r, 3), " p-value:", round(p, 4))
print()

for days, grp in cr.groupby('terms_days'):
    r, p = spearmanr(grp['late_invoice_rate'], grp['outstanding_pct'])
    print(f"Due period {int(days)} days: customers {len(grp)}, rank correlation {r:.3f}, p-value {p:.4f}")

All customers: 500  rank correlation: 0.022  p-value: 0.6283

Due period 15 days: customers 87, rank correlation 0.154, p-value 0.1556
Due period 30 days: customers 193, rank correlation 0.020, p-value 0.7834
Due period 45 days: customers 113, rank correlation 0.195, p-value 0.0384
Due period 60 days: customers 107, rank correlation 0.007, p-value 0.9447


In [42]:
p = pay_late.merge(clean_invoices[['invoice_id', 'invoice_date']], on='invoice_id', how='left')
p['payment_date'] = pd.to_datetime(p['payment_date'])
p['invoice_date'] = pd.to_datetime(p['invoice_date'])
p['due_date'] = pd.to_datetime(p['due_date'])
p['due_period'] = (p['due_date'] - p['invoice_date']).dt.days
p['days_to_pay'] = (p['payment_date'] - p['invoice_date']).dt.days
p['pay_month'] = p['payment_date'].dt.to_period('M')

last_invoice = pd.to_datetime(clean_invoices['invoice_date']).max()
print("Last invoice date:", last_invoice.date())
print("Payments used:", len(p))
print()

print("By payment month: payments, shortest days from invoice to payment, days between the last invoice and the first payment of the month")
for m, g in p[p['payment_date'] >= '2024-07-01'].groupby('pay_month'):
    gap = (g['payment_date'].min() - last_invoice).days
    print(m, len(g), g['days_to_pay'].min(), max(gap, 0))
print()

older = p[(p['payment_date'] >= '2024-07-01') & (p['payment_date'] < '2025-01-01')]
newer = p[p['payment_date'] >= '2025-01-01']
print("Payments Jul to Dec 2024:", len(older), " Payments in 2025:", len(newer))
print("By due period: [Jul to Dec 2024: payments, share %, late %, average days to pay] and [2025: payments, share %, late %, average days to pay]")
for d in sorted(p['due_period'].unique()):
    a = older[older['due_period'] == d]
    b = newer[newer['due_period'] == d]
    print(int(d),
          '[', len(a), f"{len(a) / len(older) * 100:.1f}", f"{a['is_late'].mean() * 100:.2f}", f"{a['days_to_pay'].mean():.1f}", ']',
          '[', len(b), f"{len(b) / len(newer) * 100:.1f}", f"{b['is_late'].mean() * 100:.2f}", f"{b['days_to_pay'].mean():.1f}", ']')

Last invoice date: 2025-01-13
Payments used: 18840

By payment month: payments, shortest days from invoice to payment, days between the last invoice and the first payment of the month
2024-07 240 0 0
2024-08 244 0 0
2024-09 240 0 0
2024-10 310 0 0
2024-11 237 0 0
2024-12 286 0 0
2025-01 234 0 0
2025-02 91 21 19
2025-03 17 52 47

Payments Jul to Dec 2024: 1557  Payments in 2025: 342
By due period: [Jul to Dec 2024: payments, share %, late %, average days to pay] and [2025: payments, share %, late %, average days to pay]
15 [ 232 14.9 75.86 32.7 ] [ 55 16.1 90.91 45.5 ]
30 [ 615 39.5 53.66 33.5 ] [ 144 42.1 74.31 43.6 ]
45 [ 352 22.6 29.83 33.3 ] [ 77 22.5 48.05 45.9 ]
60 [ 358 23.0 6.98 33.6 ] [ 66 19.3 7.58 43.9 ]


In [43]:
inv_p = clean_invoices[['invoice_id', 'invoice_date', 'due_date']].copy()
inv_p['invoice_date'] = pd.to_datetime(inv_p['invoice_date'])
inv_p['due_date'] = pd.to_datetime(inv_p['due_date'])

pp = payments_used.copy()
pp['payment_date'] = pd.to_datetime(pp['payment_date'])
last = pp.groupby('invoice_id')['payment_date'].max().rename('last_pay')

inv_p = inv_p.merge(last, on='invoice_id', how='left')
inv_p['has_payment'] = inv_p['last_pay'].notna()
inv_p['days_to_last'] = (inv_p['last_pay'] - inv_p['invoice_date']).dt.days
inv_p['late'] = inv_p['last_pay'] > inv_p['due_date']
inv_p['invoice_year'] = inv_p['invoice_date'].dt.year
inv_p['invoice_month'] = inv_p['invoice_date'].dt.to_period('M')

print("Invoices:", len(inv_p), " with a payment:", int(inv_p['has_payment'].sum()))
print()

print("By invoice year: invoices, with a payment, median days to last payment, average days, most days, late % (of invoices with a payment)")
for y, g in inv_p.groupby('invoice_year'):
    p = g[g['has_payment']]
    print(y, len(g), len(p), p['days_to_last'].median(), round(p['days_to_last'].mean(), 1), p['days_to_last'].max(), f"{p['late'].mean() * 100:.2f}")
print()

print("By invoice month from 2024-07: invoices, with a payment, median days, average days, late %")
for m, g in inv_p[inv_p['invoice_date'] >= '2024-07-01'].groupby('invoice_month'):
    p = g[g['has_payment']]
    print(m, len(g), len(p), p['days_to_last'].median(), round(p['days_to_last'].mean(), 1), f"{p['late'].mean() * 100:.2f}")

Invoices: 17640  with a payment: 15842

By invoice year: invoices, with a payment, median days to last payment, average days, most days, late % (of invoices with a payment)
2019 2911 2612 35.0 35.2 90.0 45.25
2020 2970 2656 36.0 35.2 90.0 45.03
2021 2952 2666 34.0 34.7 90.0 44.34
2022 2885 2578 36.0 35.7 90.0 47.40
2023 2975 2682 33.5 34.2 90.0 43.89
2024 2900 2603 37.0 36.0 90.0 46.22
2025 47 45 35.0 33.2 73.0 40.00

By invoice month from 2024-07: invoices, with a payment, median days, average days, late %
2024-07 253 226 35.0 35.4 44.25
2024-08 247 223 39.0 37.4 51.57
2024-09 242 223 36.0 35.6 45.29
2024-10 265 246 37.5 37.6 46.34
2024-11 251 232 38.0 38.0 48.71
2024-12 229 203 36.0 34.1 44.33
2025-01 47 45 35.0 33.2 40.00


In [44]:
from scipy.stats import fisher_exact, mannwhitneyu

base = inv_p[inv_p['has_payment'] & (inv_p['invoice_year'] <= 2024)]

print("Each invoice year against all the other years from 2019 to 2024")
for y in range(2019, 2025):
    a = base[base['invoice_year'] == y]
    b = base[base['invoice_year'] != y]
    t = [[int(a['late'].sum()), int((~a['late']).sum())],
         [int(b['late'].sum()), int((~b['late']).sum())]]
    print(y,
          "| average days", round(a['days_to_last'].mean(), 1), "vs", round(b['days_to_last'].mean(), 1),
          "| late %", round(a['late'].mean() * 100, 2), "vs", round(b['late'].mean() * 100, 2),
          "| p-value days", round(mannwhitneyu(a['days_to_last'], b['days_to_last']).pvalue, 4),
          "| p-value late", round(fisher_exact(t)[1], 4))
print()

early = base[base['invoice_year'] <= 2023]
recent = base[(base['invoice_date'] >= '2024-07-01') & (base['invoice_date'] < '2024-10-01')]
t = [[int(recent['late'].sum()), int((~recent['late']).sum())],
     [int(early['late'].sum()), int((~early['late']).sum())]]
print("Invoices dated Jul to Sep 2024 against 2019 to 2023")
print("  invoices with a payment:", len(recent), "vs", len(early))
print("  average days", round(recent['days_to_last'].mean(), 1), "vs", round(early['days_to_last'].mean(), 1))
print("  late %", round(recent['late'].mean() * 100, 2), "vs", round(early['late'].mean() * 100, 2))
print("  p-value days", round(mannwhitneyu(recent['days_to_last'], early['days_to_last']).pvalue, 4))
print("  p-value late", round(fisher_exact(t)[1], 4))

Each invoice year against all the other years from 2019 to 2024
2019 | average days 35.2 vs 35.2 | late % 45.25 vs 45.35 | p-value days 0.9451 | p-value late 0.9314
2020 | average days 35.2 vs 35.1 | late % 45.03 vs 45.4 | p-value days 0.9549 | p-value late 0.7325
2021 | average days 34.7 vs 35.2 | late % 44.34 vs 45.54 | p-value days 0.195 | p-value late 0.2582
2022 | average days 35.7 vs 35.1 | late % 47.4 vs 44.94 | p-value days 0.1114 | p-value late 0.0219
2023 | average days 34.2 vs 35.3 | late % 43.89 vs 45.63 | p-value days 0.0096 | p-value late 0.1012
2024 | average days 36.0 vs 35.0 | late % 46.22 vs 45.16 | p-value days 0.0185 | p-value late 0.3324

Invoices dated Jul to Sep 2024 against 2019 to 2023
  invoices with a payment: 672 vs 13194
  average days 36.1 vs 35.0
  late % 47.02 vs 45.16
  p-value days 0.1627
  p-value late 0.3609


In [45]:
raw_counts = payments['payment_id'].value_counts()
used_counts = payments_used['payment_id'].value_counts()

print("Payment rows in payments:", len(payments), " payment rows used:", len(payments_used))
print("Payment IDs repeated in payments:", int((raw_counts > 1).sum()), "covering", int(raw_counts[raw_counts > 1].sum()), "rows")
print("Payment IDs repeated in payments used:", int((used_counts > 1).sum()), "covering", int(used_counts[used_counts > 1].sum()), "rows")

repeated_raw = raw_counts[raw_counts > 1].index
single_now = [i for i in repeated_raw if used_counts.get(i, 0) == 1]
missing_now = [i for i in repeated_raw if used_counts.get(i, 0) == 0]
print("Repeated in payments but single in payments used:", len(single_now))
print("Repeated in payments and missing from payments used:", len(missing_now))
print()

with_customer = payments_used[['payment_id', 'invoice_id']].merge(
    clean_invoices[['invoice_id', 'customer_id']], on='invoice_id', how='left')
shared = with_customer[with_customer['payment_id'].isin(used_counts[used_counts > 1].index)]
per_id = shared.groupby('payment_id').agg(rows=('invoice_id', 'size'),
                                          invoices=('invoice_id', 'nunique'),
                                          customers=('customer_id', 'nunique'))

print("Shared payment IDs in payments used:", len(per_id))
print("Rows without a customer:", int(shared['customer_id'].isna().sum()))
print("Rows per shared ID:", per_id['rows'].value_counts().to_dict())
print("IDs whose rows point to different invoices:", int((per_id['invoices'] > 1).sum()))
print("IDs whose rows belong to one customer:", int((per_id['customers'] == 1).sum()))
print("IDs whose rows belong to different customers:", int((per_id['customers'] > 1).sum()))

Payment rows in payments: 19257  payment rows used: 18840
Payment IDs repeated in payments: 201 covering 403 rows
Payment IDs repeated in payments used: 193 covering 387 rows
Repeated in payments but single in payments used: 8
Repeated in payments and missing from payments used: 0

Shared payment IDs in payments used: 193
Rows without a customer: 0
Rows per shared ID: {2: 192, 3: 1}
IDs whose rows point to different invoices: 193
IDs whose rows belong to one customer: 1
IDs whose rows belong to different customers: 192


In [46]:
one_customer_ids = per_id[per_id['customers'] == 1].index

show = payments_used[payments_used['payment_id'].isin(one_customer_ids)].merge(
    clean_invoices[['invoice_id', 'customer_id', 'invoice_date', 'grand_total']].rename(
        columns={'invoice_date': 'invoice_date_of_invoice', 'grand_total': 'invoice_total'}
    ),
    on='invoice_id',
    how='left'
)

print("Payment IDs with one customer:", len(one_customer_ids))
print(show.to_string(index=False))

Payment IDs with one customer: 1
payment_id invoice_id payment_date  payment_amount payment_method customer_id invoice_date_of_invoice  invoice_total
PAY-366706 INV-134522   2019-02-21        56548.55            UPI       C0475              2019-02-08        91273.0
PAY-366706 INV-882734   2024-03-08        48179.40           Cash       C0475              2024-01-13        48179.4


In [47]:
print(clean_invoices.columns.tolist())
print(clean_invoices[clean_invoices['customer_id'] == 'C0297'].head(2).to_string(index=False))

['invoice_id', 'so_id', 'customer_id', 'branch_id', 'invoice_date', 'due_date', 'total_order_value', 'total_gst_amount', 'grand_total', 'payment_status']
invoice_id     so_id customer_id branch_id invoice_date   due_date  total_order_value  total_gst_amount  grand_total payment_status
INV-437984 SO-442942       C0297    CHN001   2019-11-12 2019-11-27            2618440          728634.2    3347074.2 Partially Paid
INV-733889 SO-684716       C0297    CHN001   2023-09-24 2023-10-09            2796310          779315.8    3575625.8           Paid


In [48]:
c_inv = clean_invoices[clean_invoices['customer_id'] == 'C0297'][['invoice_id', 'due_date', 'payment_status']].copy()
c_inv['due_date'] = pd.to_datetime(c_inv['due_date'])

c_pay = payments_used[payments_used['invoice_id'].isin(c_inv['invoice_id'])].copy()
c_pay['payment_date'] = pd.to_datetime(c_pay['payment_date'])
c_pay = c_pay.merge(c_inv[['invoice_id', 'due_date']], on='invoice_id', how='left')
c_pay['paid_after_due'] = c_pay['payment_date'] > c_pay['due_date']

per_invoice = c_pay.groupby('invoice_id').agg(
    payments=('payment_id', 'count'),
    first_payment=('payment_date', 'min'),
    last_payment=('payment_date', 'max'),
    late_payments=('paid_after_due', 'sum')
).reset_index()

paid_rows = c_inv[c_inv['payment_status'] == 'Paid'].merge(per_invoice, on='invoice_id', how='left')
paid_rows['days_last_payment_after_due'] = (paid_rows['last_payment'] - paid_rows['due_date']).dt.days

print(paid_rows.sort_values('due_date').to_string(index=False))
print()
print("Paid invoices:", len(paid_rows))
print("Paid invoices with a payment:", paid_rows['payments'].notna().sum())
print("Paid invoices with the last payment after the due date:", (paid_rows['days_last_payment_after_due'] > 0).sum())
print("Paid invoices with at least one payment after the due date:", (paid_rows['late_payments'] > 0).sum())
print("Whole customer, invoices with at least one payment after the due date:", (per_invoice['late_payments'] > 0).sum())
print("Whole customer, payments after the due date:", int(c_pay['paid_after_due'].sum()))

invoice_id   due_date payment_status  payments first_payment last_payment  late_payments  days_last_payment_after_due
INV-954881 2019-03-31           Paid         2    2019-05-06   2019-05-07              2                           37
INV-371181 2019-08-06           Paid         1    2019-09-12   2019-09-12              1                           37
INV-546306 2020-10-27           Paid         1    2020-10-30   2020-10-30              1                            3
INV-695209 2020-12-04           Paid         1    2020-12-10   2020-12-10              1                            6
INV-898040 2021-02-05           Paid         2    2021-01-31   2021-03-21              1                           44
INV-644291 2021-06-20           Paid         2    2021-06-20   2021-07-06              1                           16
INV-898706 2021-06-26           Paid         1    2021-07-31   2021-07-31              1                           35
INV-295117 2022-04-02           Paid         1    2022-0

In [49]:
last_pay = payments_used.assign(payment_date=pd.to_datetime(payments_used['payment_date'])).groupby('invoice_id')['payment_date'].max().reset_index()

pace = clean_invoices[['invoice_id', 'invoice_date', 'due_date']].merge(last_pay, on='invoice_id', how='inner')
pace['invoice_date'] = pd.to_datetime(pace['invoice_date'])
pace['due_date'] = pd.to_datetime(pace['due_date'])
pace['due_period'] = (pace['due_date'] - pace['invoice_date']).dt.days
pace['days_to_last_payment'] = (pace['payment_date'] - pace['invoice_date']).dt.days

from scipy import stats as sps
from itertools import combinations

periods = [15, 30, 45, 60]
groups = {k: pace.loc[pace['due_period'] == k, 'days_to_last_payment'] for k in periods}

print("Invoices with a payment:", len(pace))
for k in periods:
    g = groups[k]
    print("Due period", k, "days: invoices", len(g), "median", g.median(), "average", round(g.mean(), 2))

h, p = sps.kruskal(*groups.values())
print("All four due periods compared, p-value:", round(p, 4))

print("Pairs, strict line 0.0083:")
for a, b in combinations(periods, 2):
    p_pair = sps.mannwhitneyu(groups[a], groups[b], alternative='two-sided').pvalue
    print(a, "days vs", b, "days: p-value", round(p_pair, 4))

Invoices with a payment: 15842
Due period 15 days: invoices 2772 median 34.0 average 34.55
Due period 30 days: invoices 6195 median 35.0 average 35.21
Due period 45 days: invoices 3549 median 35.0 average 35.25
Due period 60 days: invoices 3326 median 36.0 average 35.44
All four due periods compared, p-value: 0.3861
Pairs, strict line 0.0083:
15 days vs 30 days: p-value 0.1979
15 days vs 45 days: p-value 0.2102
15 days vs 60 days: p-value 0.0919
30 days vs 45 days: p-value 0.9211
30 days vs 60 days: p-value 0.5092
45 days vs 60 days: p-value 0.6143
